In [1]:
import sys
sys.path.append(r'E:\ipynb')


import pandas as pd
import os
import datetime
import numpy as np
import pymysql
from tqdm import tqdm
from itertools import product
import warnings
warnings.filterwarnings('ignore')
import math
from sqlalchemy import create_engine
from sqlalchemy.types import NVARCHAR, Float, Integer
from password import passowrd_m,passowrd_c



def sql_to_data(sql,db_name='yibai_order',args=passowrd_m['139.159.188.220']):
#     print(sql)
    conn = pymysql.connect(host=args['host'], user=args['user'],password=args['password'],database=db_name,charset=args['charset'],port=args['port'])
    
    cursor = conn.cursor()
    cursor.execute(sql)
    data=cursor.fetchall()
    data=pd.DataFrame(data,columns=[col[0] for col in cursor.description])
    cursor.close()
    conn.close()
    
    return data



# author: marmot
from clickhouse_driver import Client
import pandas as pd
import datetime
from pandahouse import read_clickhouse

class NewPandaHouse(object):
    def __init__(self, user="default", password="", host="localhost", port=8123, db_name="default", db_sheet=""):
        self.host, self.port = host, port
        self.db_name, self.db_sheet = db_name, db_sheet
        self.client = Client(host=self.host, port=self.port, user=user, password=password, compression=True)
        
    def ck_select_to_df(self, ck_sql=None, columns=None):
        """
        :param ck_sql : clickhouse的sql语句
        :param columns : list 返回的df的列名,为空则没有列名
        """
        answer = self.client.execute(ck_sql, with_column_types=True)
        data_list = answer[0]
        answer_df = pd.DataFrame(data_list)
        columns_result_list = [one_item[0] for one_item in answer[1]]
        if columns_result_list and data_list:
            answer_df.columns = columns_result_list
            if columns:
                answer_df = answer_df[columns]
                
        return answer_df

    
def get_data(sql,args=passowrd_c['124.71.13.250'],db_name='send_fba'):
#     print(sql)
    ck_write_cls = NewPandaHouse(user=args['user'], password=args['password'], host=args['host'], port=args['port'],db_name=db_name)    
    yibai_price_adjust_log_v3 = ck_write_cls.ck_select_to_df(sql)
    return  yibai_price_adjust_log_v3


In [2]:
sql_account="""
select a.account_id '账号ID',a.account_status '账号状态',a.account_id '平台账号ID',a.account_name '账号全称',a.account_s_name'账号简称',
a.account_num_name '账号店铺名',a.account_operation_mode '账号运营模式' ,c.main_body_name,d.body_org_name,a.oa_group_name
from yibai_account_manage.yibai_temu_account_v a
left join yibai_account_manage.yibai_account_main_info c
on a.main_code=c.main_body_code
left join yibai_account_manage.yibai_account_org_info d
on c.main_body_code=d.main_body_code and a.org_code=d.body_org_code
where a.account_type=1
"""
print(sql_account)
account=sql_to_data(sql_account,db_name='yibai_account_manage',args=passowrd_m['121.37.228.71'])
account['账号状态']=account['账号状态'].map({10:'启用',20:'停用',30:'异常',})
account['账号运营模式']=account['账号运营模式'].map({1:'全托管',2:'半托管'})
account.rename(columns={'账号ID':'account_id'},inplace=True)
account.rename(columns={'main_body_name':'主体账号','body_org_name':'组织账号'},inplace=True)
account['主体账号']=account['组织账号'].map({
'易佰精铺Temu2组（杨静）':'杨静',
'易佰泛品：李金强':'李金强',
'易佰泛品Temu1组（Libby）':'libby',
'易佰合伙人1006Temu9组（子龙）':'刘子龙'})
# account.to_excel(r'C:\Users\Administrator\Desktop\Temu账号.xlsx',index=False)


select a.account_id '账号ID',a.account_status '账号状态',a.account_id '平台账号ID',a.account_name '账号全称',a.account_s_name'账号简称',
a.account_num_name '账号店铺名',a.account_operation_mode '账号运营模式' ,c.main_body_name,d.body_org_name,a.oa_group_name
from yibai_account_manage.yibai_temu_account_v a
left join yibai_account_manage.yibai_account_main_info c
on a.main_code=c.main_body_code
left join yibai_account_manage.yibai_account_org_info d
on c.main_body_code=d.main_body_code and a.org_code=d.body_org_code
where a.account_type=1



In [3]:
account.head(1)

,account_id,账号状态,平台账号ID,账号全称,账号简称,账号店铺名,账号运营模式,主体账号,组织账号,oa_group_name
0,339,启用,339,TEMU-Civilizationa,CV30,TEMU-Civilizationa,半托管,杨静,易佰精铺Temu2组（杨静）,Temu2组（杨静）


In [4]:
account[['主体账号','组织账号']].drop_duplicates().sort_values(by=['主体账号'])

,主体账号,组织账号
15,libby,易佰泛品Temu1组（Libby）
22,刘子龙,易佰合伙人1006Temu9组（子龙）
77,李金强,易佰泛品：李金强
0,杨静,易佰精铺Temu2组（杨静）
16,NaN,易佰控股-小满科技
56,NaN,None
57,NaN,VC专项组
58,NaN,temu部
525,NaN,七部亚马逊
933,NaN,易佰控股-易通兔


In [5]:
# Temu的刊登覆盖率

# 1.获取连接
# --  `lazada_account_operation_mode` tinyint(4) NULL COMMENT "lazada账号运营模式 1自运营  全托管   2托管  半托管",
# yibai_temu_listing_crawling_log加入站点时间 added_to_site_time
#   yibai_temu_listing_crawling_log_zt.`supplier_price` double NULL COMMENT "申报价格",
#   yibai_temu_listing_zt.`create_time` datetime NULL COMMENT "刊登时间",

sql="""
with f as 
(select id,product_spu_id,product_sku_id,online_status,added_to_site_time,supplier_price from yibai_product.yibai_temu_listing_crawling_log_other_org
union all 
select id,product_spu_id,product_sku_id,online_status,added_to_site_time,supplier_price from yibai_product.yibai_temu_listing_crawling_log_zt),
d as
(select product_spu_id,product_sku_id,max(id) as id from f
group by product_spu_id,product_sku_id),
c as (select f.* from f inner join d on f.id=d.id)
select e.account_id,e.short_name,a.site_code,a.item_id,a.product_sku_id,a.product_skc_id,a.stock_number,c.online_status,a.sku,b.lazada_account_operation_mode,
c.added_to_site_time,c.supplier_price,date(a.create_time) as '刊登时间',a.last_update_time,a.select_status from yibai_product.yibai_temu_listing_zt a
left join yibai_system.yibai_common_account_config b on a.account_id=b.account_id
left join c on a.item_id =c.product_spu_id and a.product_sku_id=c.product_sku_id
left join yibai_system.yibai_system_account_v as e
on a.account_id=e.id
where e.platform_code='TEMU' and e.is_del=0 and b.is_del=0
"""

print(sql)
listing_t=sql_to_data(sql,db_name='yibai_product',args=passowrd_m['121.37.228.71'])

# 2025.1.24 
del listing_t['supplier_price']
# 申报价格取值-> yibai_sale_center_temu.yibai_temu_listing_supplier_price_site.supplier_price

# 链接状态：
# 优先取yibai_temu_listing.select_status
# 若yibai_temu_listing.select_status=-1，则取yibai_temu_listing_crawling_log.online_status
# 和yibai_temu_listing_crawling_log_other_or.online_status 最新的值

#  `select_status` bigint(20) NULL COMMENT "价格申报状态 -1未同步 0已弃用 1待平台选品 14待卖家修改 15已修改 16服饰可加色 
#     2待上传生产资料 3待寄样 4寄样中 5待平台审版 6审版不合格 7平台核价中 8待修改生产资料 9核价未通过 10待下首单 11已下首单 12已加入站点 
# 13已下架 17已终止",
listing_t['select_status1']=listing_t['select_status'].map({-1: '未知',0: '已弃用',1: '待平台选品',14: '待卖家修改',15: '已修改',
16: '服饰可加色',2: '待上传生产资料',3: '待寄样',4: '寄样中',5: '待平台审版',6: '审版不合格',7: '平台核价中',8: '待修改生产资料',
9: '核价未通过',10: '待下首单',11: '已下首单',12: '已加入站点',13: '已下架',17: '已终止'})
listing_t[['select_status','select_status1']].drop_duplicates().sort_values(by=['select_status'])
# listing_t['online_status']=np.where(listing_t['select_status']==-1,listing_t['online_status'],listing_t['select_status1'])
listing_t['online_status']=listing_t['select_status1']


listing_t['运营模式']=listing_t['lazada_account_operation_mode'].map({1:'全托管',2:'半托管'})
del listing_t['lazada_account_operation_mode']
listing_t=listing_t[listing_t['运营模式']=='半托管']


listing_t=listing_t.merge(account[['account_id','主体账号']],on=['account_id'],how='left')

country={'us': '美国','it': '欧洲','sp': '欧洲','fr': '欧洲','de': '欧洲','uk': '英国','jp': '日本','ca': '加拿大','mx': '墨西哥','au': '澳大利亚','in': '印度','ae': '阿联酋','sg': '新加坡','nl': '欧洲','br': '巴西','sa': '沙特','se': '欧洲','pl': '欧洲','tr': '欧洲','be':'欧洲','es':'欧洲','gb':'英国','eu':'欧洲','cz':'欧洲','hu':'欧洲','nz':'新西兰','pt':'欧洲','ro':'欧洲','at':'欧洲','dk':'欧洲','ph':'菲律宾','sk':'欧洲'}
listing_t['站点']=listing_t['site_code'].str.split(',').str[0].str.lower().map(country)
listing_t[['site_code','站点']].drop_duplicates()


listing_t['sku']=listing_t['sku'].astype(str).str.strip()
listing_t.rename(columns={'added_to_site_time':'加入站点时间','supplier_price':'申报价格'},inplace=True)
print(listing_t[listing_t['站点'].isnull()][['site_code','站点']].drop_duplicates())
listing_t['product_sku_id']=listing_t['product_sku_id'].astype(str)

listing_t['刊登时间']=pd.to_datetime(listing_t['刊登时间'],format='%Y-%m-%d')
listing_t.sort_values(by=['last_update_time'],ascending=False,inplace=True)
listing_t.drop_duplicates(subset=['account_id','product_sku_id'],inplace=True)

# #2024.12.27
# listing_t['备注']=np.where((~listing_t['site_code'].str.contains('DE|ES|FR|IT|SP'))&(listing_t['站点']=='欧洲'),'不统计','统计')



with f as 
(select id,product_spu_id,product_sku_id,online_status,added_to_site_time,supplier_price from yibai_product.yibai_temu_listing_crawling_log_other_org
union all 
select id,product_spu_id,product_sku_id,online_status,added_to_site_time,supplier_price from yibai_product.yibai_temu_listing_crawling_log_zt),
d as
(select product_spu_id,product_sku_id,max(id) as id from f
group by product_spu_id,product_sku_id),
c as (select f.* from f inner join d on f.id=d.id)
select e.account_id,e.short_name,a.site_code,a.item_id,a.product_sku_id,a.product_skc_id,a.stock_number,c.online_status,a.sku,b.lazada_account_operation_mode,
c.added_to_site_time,c.supplier_price,date(a.create_time) as '刊登时间',a.last_update_time,a.select_status from yibai_product.yibai_temu_listing_zt a
left join yibai_system.yibai_common_account_config b on a.account_id=b.account_id
left join c on a.item_id =c.product_spu_id and a.product_sku_id=c.product_sku_id
left join yibai_system.yibai_system_account_v as e
on a.acc

In [6]:
listing_t.head(1)

,account_id,short_name,site_code,item_id,product_sku_id,product_skc_id,stock_number,online_status,sku,加入站点时间,刊登时间,last_update_time,select_status,select_status1,运营模式,主体账号,站点
2270234,2324,SY37,CZ,415144432,5613202892,6067107657,0,已加入站点,JYB01688-01,2024-11-20 14:23:02,2024-11-17,2025-09-01 05:51:46,12,已加入站点,半托管,李金强,欧洲


In [7]:
len(listing_t)

2674134

In [8]:
# 第一个  针对在线listing无对应公司sku的通过sku绑定关系的数据补充
# 第二个  但凡是涉及分组织的数据  不要用大部小组两个字段去分，命名太乱了，暂用主体账号这个字段去分组织就行 

sql1="""
select erp_id as account_id,platform_sku as product_sku_id,company_sku as sku from yibai_product.yibai_temu_bind_sku_zt
order by update_time desc
"""
yibai_temu_bind_sku=sql_to_data(sql1,db_name='yibai_product',args=passowrd_m['121.37.228.71'])
yibai_temu_bind_sku.drop_duplicates(inplace=True)
yibai_temu_bind_sku.drop_duplicates(subset=['account_id','product_sku_id'],inplace=True)

listing_t=listing_t.merge(yibai_temu_bind_sku,on=['account_id','product_sku_id'],how='left',suffixes=['','1'])
listing_t.loc[listing_t['sku']=='','sku']=np.nan
listing_t['sku'].fillna(listing_t['sku1'],inplace=True)


In [9]:
listing_t.head(1)

,account_id,short_name,site_code,item_id,product_sku_id,product_skc_id,stock_number,online_status,sku,加入站点时间,刊登时间,last_update_time,select_status,select_status1,运营模式,主体账号,站点,sku1
0,2324,SY37,CZ,415144432,5613202892,6067107657,0,已加入站点,JYB01688-01,2024-11-20 14:23:02,2024-11-17,2025-09-01 05:51:46,12,已加入站点,半托管,李金强,欧洲,JYB01688-01


In [10]:
#2024.10.31 精铺+精品团队链接部分在sass系统捆绑
# yibai_dcm_base.dcm_product  “我的商品列表” 类目和易佰一样  yibai_dcm_base.dcm_category
# yibai_dcm_order.dcm_order  订单主表->独立的，他们不会在刊登维护账号信息   无需想多组织订单一样进行转化
# dcm_bind_distributor_dmstemu.shop_id  is_del 10=未删除 20=已删除

sql="""
select b.short_name,b.account,a.seller_sku,a.dcm_sku,a.is_del  from jp_saas_erp_base.dcm_bind_distributor_dmstemu a
left join jp_saas_erp_base.dcm_shop_v b on a.shop_id=b.id and a.distributor_id=b.distributor_id
"""

dcm_bind_distributor_dmstemu=sql_to_data(sql,db_name='jp_saas_erp_base',args=passowrd_m['121.37.228.71'])
dcm_bind_distributor_dmstemu['short_name']=dcm_bind_distributor_dmstemu['short_name'].str.strip('\-| ')
dcm_bind_distributor_dmstemu=dcm_bind_distributor_dmstemu.merge(account[['账号简称','account_id','账号全称','主体账号']].rename(columns={'账号简称':'short_name'}),on=['short_name'],how='left')
print(len(dcm_bind_distributor_dmstemu[dcm_bind_distributor_dmstemu['account_id'].isnull()]))
dcm_bind_distributor_dmstemu[dcm_bind_distributor_dmstemu['account']!=dcm_bind_distributor_dmstemu['账号全称']]
dcm_bind_distributor_dmstemu.rename(columns={'seller_sku':'product_sku_id','dcm_sku':'sku'},inplace=True)
dcm_bind_distributor_dmstemu.sort_values(by=['account_id','product_sku_id','is_del'],inplace=True)
dcm_bind_distributor_dmstemu.drop_duplicates(subset=['account_id','product_sku_id'],inplace=True)

listing_t=listing_t.merge(dcm_bind_distributor_dmstemu[['account_id','product_sku_id','sku']].drop_duplicates(),on=['account_id','product_sku_id'],how='left',suffixes=['','2'])
listing_t.loc[listing_t['sku']=='','sku']=np.nan
listing_t['sku'].fillna(listing_t['sku2'],inplace=True)
# listing_t[listing_t['sku2'].notnull()][['short_name','product_sku_id','sku','主体账号']]

#2025.4.14 精品团队捆绑sku从sass系统同步到中台，sass系统捆绑为系统SKU，同步到中台的为商户SKU
sql="""
select sku,custom_sku from jp_saas_erp_base.dcm_my_product_distributor_alls where sku !='' and is_del=10
"""
dcm_my_product_distributor_alls=sql_to_data(sql,db_name='jp_saas_erp_base',args=passowrd_m['121.37.228.71'])
listing_t=listing_t.merge(dcm_my_product_distributor_alls.rename(columns={'custom_sku':'sku','sku':'系统sku'}),on=['sku'],how='left')
listing_t.loc[(listing_t['主体账号']=='李茜')&listing_t['系统sku'].notnull(),'sku']=listing_t.loc[(listing_t['主体账号']=='李茜')&listing_t['系统sku'].notnull(),'系统sku']


0


In [11]:
dcm_bind_distributor_dmstemu.head(1)

,short_name,account,product_sku_id,sku,is_del,account_id,账号全称,主体账号
3213,RI19,TEMU-Rascxzi,1025428336,YM12104YBJPS23B00013,10,85,TEMU-Rascxzi,NaN


In [12]:
listing_t.head(1)

,account_id,short_name,site_code,item_id,product_sku_id,product_skc_id,stock_number,online_status,sku,加入站点时间,刊登时间,last_update_time,select_status,select_status1,运营模式,主体账号,站点,sku1,sku2,系统sku
0,2324,SY37,CZ,415144432,5613202892,6067107657,0,已加入站点,JYB01688-01,2024-11-20 14:23:02,2024-11-17,2025-09-01 05:51:46,12,已加入站点,半托管,李金强,欧洲,JYB01688-01,NaN,NaN


In [13]:
listing_t['online_status'].value_counts()

待下首单     801603
已加入站点    530862
平台核价中    372604
核价未通过    346257
已下架      256485
待平台选品    227441
已终止       79306
已下首单      30965
未知        29822
Name: online_status, dtype: int64

In [14]:
listing_t1=listing_t.copy()
listing_t1.loc[listing_t1['online_status']=='','online_status']='待申报'
listing_t1.loc[listing_t1['online_status'].isnull(),'online_status']='未知'
listing_t1['是否核价通过链接']=np.where(listing_t1['online_status'].isin(['已加入站点','已下架','已终止']),1,0)
listing_t1['product_sku_id']=listing_t1['product_sku_id'].astype(str)
listing_t1['加入站点时间']=pd.to_datetime(listing_t1['加入站点时间'].dt.date,format='%Y-%m-%d')
listing_t1['是否已发布到站点']=np.where(listing_t1['online_status'].isin(['已加入站点']),1,0)

In [15]:
listing_t1.head(1)

,account_id,short_name,site_code,item_id,product_sku_id,product_skc_id,stock_number,online_status,sku,加入站点时间,...,select_status,select_status1,运营模式,主体账号,站点,sku1,sku2,系统sku,是否核价通过链接,是否已发布到站点
0,2324,SY37,CZ,415144432,5613202892,6067107657,0,已加入站点,JYB01688-01,2024-11-20,...,12,已加入站点,半托管,李金强,欧洲,JYB01688-01,NaN,NaN,1,1


In [16]:
set(listing_t1['online_status'])

{'已下架', '已下首单', '已加入站点', '已终止', '平台核价中', '待下首单', '待平台选品', '未知', '核价未通过'}

In [17]:
# ###20250704
# A=listing_t1[listing_t1['是否已发布到站点']==1].groupby(['short_name','主体账号'],dropna=False).agg({'account_id':'count'})
# A.rename(columns={'account_id':'加站链接数'},inplace=True)
# A.to_excel(r'C:\Users\Administrator\Desktop\Temu加站链接数2025-07-04.xlsx')

In [18]:
#去仓标  2024.8.9

d=pd.read_excel(r"E:\20240805-Temu团队业绩结构分析\海外仓主子SKU映射关系维护.xlsx")
d['海外仓SKU']=d['海外仓SKU'].astype(str)
d['国内仓主SKU']=d['国内仓主SKU'].astype(str)

# 仓标   
warehouse_code=['US','AU','DE','GB','FR','IT','ES','CA']
d['仓标']=d['海外仓SKU'].str.findall('|'.join(warehouse_code)).str[0]
d['数量']=d['海外仓SKU'].str.findall('|'.join(warehouse_code)).str.len()
print(set(d['数量']))


{1, 2}


In [19]:
d.head(1)

,海外仓SKU,国内仓主SKU,仓标,数量
0,US-00EGS06800,00EGS06800,US,1


In [20]:
#第十版责任类目

#确认责任类目  2024.9.20
group_names={'Temu4组': '李金强',
'Temu3组': '李金强',
 'Temu5组': '李金强',
 'Temu11组': '李金强',
 'Temu1组': 'libby',
 'Temu合伙制': '刘子龙'}

path1=pd.read_excel(r"E:\20240712-Temu\易佰Temu盲发+海外仓常规责任类目划分-第二十二版25.7.31-职能部门.xlsx",sheet_name='盲发---批次+SKU+国家+责任组明细',usecols=['sku','站点','第二十二版-责任组'])
# path1.rename(columns={'第二十版-责任人':'责任人'},inplace=True)
path1['责任人']=path1['第二十二版-责任组'].map(group_names)
del path1['第二十二版-责任组']

path2=pd.read_excel(r"E:\20240712-Temu\易佰Temu盲发+海外仓常规责任类目划分-第二十二版25.7.31-职能部门.xlsx",sheet_name='海外仓常规---SKU+国家+责任组明细',usecols=['SKU','国家','第二十二版-责任组'])
path2.rename(columns={'SKU':'sku','国家':'站点'},inplace=True)
path2['责任人']=path2['第二十二版-责任组'].map(group_names)
del path2['第二十二版-责任组']

path3=pd.read_excel(r"E:\20240712-Temu\易佰Temu盲发+海外仓常规责任类目划分-第二十二版25.7.31-职能部门.xlsx",sheet_name='精铺大部',skiprows=1,usecols=['Saas SKU','目的国','第十五版-责任人1'])
path3.rename(columns={'Saas SKU':'sku','目的国':'站点','第十五版-责任人1':'责任人'},inplace=True)
path3=path3[path3['责任人'].notnull()]


path31=pd.read_excel(r"E:\20240712-Temu\易佰Temu盲发+海外仓常规责任类目划分-第二十二版25.7.31-职能部门.xlsx",sheet_name='精铺大部',skiprows=1,usecols=['Saas SKU','目的国','第十五版-责任人2'])
path31.rename(columns={'Saas SKU':'sku','目的国':'站点','第十五版-责任人2':'责任人'},inplace=True)
path31=path31[path31['责任人'].notnull()]

path4=pd.read_excel(r"E:\20240712-Temu\易佰Temu盲发+海外仓常规责任类目划分-第二十二版25.7.31-职能部门.xlsx",sheet_name='精品',usecols=['系统SKU','目的国','责任人'])
path4.rename(columns={'系统SKU':'sku','目的国':'站点'},inplace=True)
# path4.loc[path4['责任人']=='李茜','责任人']='茜茜子'

path=pd.concat([path1,path2,path3,path31,path4])
path['sku']=path['sku'].astype(str).str.strip()
path.drop_duplicates(inplace=True)
print(set(path['站点']))
path.loc[path['责任人']=='刘子龙','责任人']='子龙'


A=account[account['主体账号'].notnull()][['主体账号']].drop_duplicates()
A['责任人']=A['主体账号'].str.findall('|'.join(set(path['责任人']))).str[0]
path=path.merge(A,on=['责任人'],how='left')
print(path[['责任人','主体账号']].drop_duplicates())


#去仓标
path=path.merge(d[['海外仓SKU','国内仓主SKU']].rename(columns={'海外仓SKU':'sku'}),on=['sku'],how='left')
path['仓标']=path['sku'].str.findall('|'.join(warehouse_code)).str[0]

import re
def f1(x):
    if x['仓标'] is not np.nan:
        a=x['仓标']
        b=x['仓标']+'-'
        c='-'+x['仓标']
        
        sku=x['sku']
        A=set([a,b,c])&set([sku[:2],sku[:3],sku[-2:],sku[-3:]])
        if len(A)>0:
            return x['sku'].replace(b,'').replace(c,'').replace(a,'')
        else:
            return x['sku']

path['国内仓sku']=np.where(path['国内仓主SKU'].notnull(),path['国内仓主SKU'],np.where(path['仓标'].isnull(),path['sku'],path.apply(f1,axis=1)))
del path['国内仓主SKU']
del path['仓标']
path['是否责任类目']='是'


{'英国', '欧洲', '美国', '加拿大', '澳大利亚'}
         责任人   主体账号
0        李金强    李金强
165    libby  libby
458       子龙    刘子龙
44852     杨静     杨静
45898     李茜    NaN


In [21]:
path.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目
0,英国,GS21061,李金强,李金强,GS21061,是


In [23]:
#2024.11.1  排除销售提报不上架的sku->群"TEMU不上架产品"

d1=pd.read_excel(r"E:\20240712-Temu\易佰-Temu无法刊登的海外仓SKU汇总.xlsx",usecols=['SKU','目的国\n(按照下拉框格式填写)'],sheet_name='易佰')
d1.rename(columns={'SKU':'sku','目的国\n(按照下拉框格式填写)':'站点'},inplace=True)
d1['sku']=d1['sku'].astype(str).str.strip()
print(set(d1['站点']))
d1['是否可上架']='否'
d1.loc[d1['站点']=='澳洲','站点']='澳大利亚'
d1.loc[d1['站点']=='法国','站点']='欧洲'
d1.loc[d1['站点']=='德国','站点']='欧洲'
d1.drop_duplicates(inplace=True)
path=path.merge(d1,on=['sku','站点'],how='left')
path['是否可上架'].fillna('是',inplace=True)
print(path['是否可上架'].value_counts())

{'英国', '欧洲', '美国', '加拿大', '澳大利亚'}
是    37902
否     8211
Name: 是否可上架, dtype: int64


In [24]:
path.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架
0,英国,GS21061,李金强,李金强,GS21061,是,是


In [25]:
listing_t1.head(1)

,account_id,short_name,site_code,item_id,product_sku_id,product_skc_id,stock_number,online_status,sku,加入站点时间,...,select_status,select_status1,运营模式,主体账号,站点,sku1,sku2,系统sku,是否核价通过链接,是否已发布到站点
0,2324,SY37,CZ,415144432,5613202892,6067107657,0,已加入站点,JYB01688-01,2024-11-20,...,12,已加入站点,半托管,李金强,欧洲,JYB01688-01,NaN,NaN,1,1


In [26]:
#拆分捆绑sku  确认责任sku是否已刊登  是否加站成功

sku=listing_t1[listing_t1['sku'].notnull()][['sku','站点','主体账号','是否核价通过链接','是否已发布到站点']].groupby(['sku','站点','主体账号']).agg({'是否核价通过链接':['sum','count','max'],'是否已发布到站点':['sum','max']})
sku.columns=[i+j for i,j in sku.columns]
# .droplevel(axis=1,level=0).rename(columns={'sum':'加站链接数','count':'刊登链接数','max':'是否核价通过链接'}).reset_index()
sku.rename(columns={'是否核价通过链接sum':'核价通过链接数','是否核价通过链接count':'刊登链接数','是否核价通过链接max':'是否核价通过链接','是否已发布到站点sum':'加站链接数','是否已发布到站点max':'是否已发布到站点'},inplace=True)
sku.reset_index(inplace=True)
sku.index=range(len(sku))
sku=sku.join(sku['sku'].str.split('+',expand=True).stack().reset_index(level=1,drop=True).str.split('\*',expand=True).rename(columns={0:'拆分后sku',1:'数量'}))
del sku['数量']
sku=sku[['主体账号','sku','站点','拆分后sku','核价通过链接数','是否核价通过链接','加站链接数','是否已发布到站点','刊登链接数']].drop_duplicates()


sku=sku.merge(d[['海外仓SKU','国内仓主SKU']].rename(columns={'海外仓SKU':'拆分后sku'}),on=['拆分后sku'],how='left')
sku['仓标']=sku['拆分后sku'].str.findall('|'.join(warehouse_code)).str[0]


import re
def f1(x):
    if x['仓标'] is not np.nan:
        a=x['仓标']
        b=x['仓标']+'-'
        c='-'+x['仓标']
        
        sku=x['拆分后sku']
        A=set([a,b,c])&set([sku[:2],sku[:3],sku[-2:],sku[-3:]])
        if len(A)>0:
            return x['拆分后sku'].replace(b,'').replace(c,'').replace(a,'')
        else:
            return x['拆分后sku']

sku['国内仓sku']=np.where(sku['国内仓主SKU'].notnull(),sku['国内仓主SKU'],np.where(sku['仓标'].isnull(),sku['拆分后sku'],sku.apply(f1,axis=1)))
del sku['国内仓主SKU']
del sku['仓标']

sku=sku.groupby(['国内仓sku','站点','主体账号']).agg({'是否核价通过链接':'max','是否已发布到站点':'max','核价通过链接数':'sum','加站链接数':'sum','刊登链接数':'sum'}).reset_index()
sku['是否已刊登']='是'


In [27]:
sku.head(1)

,国内仓sku,站点,主体账号,是否核价通过链接,是否已发布到站点,核价通过链接数,加站链接数,刊登链接数,是否已刊登
0,\n2717250013512,美国,刘子龙,1,0,1,0,1,是


In [28]:
path1=path.merge(sku[['国内仓sku','站点','主体账号','是否已刊登','是否核价通过链接','是否已发布到站点','核价通过链接数','加站链接数','刊登链接数']],on=['国内仓sku','站点','主体账号'],how='left')\
# .merge(sku[['拆分后sku','站点','主体账号','是否已刊登']].rename(columns={'拆分后sku':'sku'}).drop_duplicates(),on=['sku','站点','主体账号'],how='left',suffixes=['','1'])
print(len(path1)-len(path))
path1['是否已刊登'].fillna('否',inplace=True)
# path1['是否已刊登1'].fillna('否',inplace=True)


0


In [29]:
path1.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架,是否已刊登,是否核价通过链接,是否已发布到站点,核价通过链接数,加站链接数,刊登链接数
0,英国,GS21061,李金强,李金强,GS21061,是,是,是,1.0,1.0,57.0,27.0,87.0


In [30]:
len(path1)

46113

In [31]:
path1['是否已刊登'].value_counts()

是    41400
否     4713
Name: 是否已刊登, dtype: int64

In [32]:
#判断当前是否有库存
# 1、不区分是否有库存
# 2、可用库存>0
# 3、可用库存>3

# PS：仓库范围：查询易佰海外仓公共仓+Temu独享仓

#易佰temu可用库存  公共仓+Temu独享仓
# 易佰  temu   可以用   易佰公共仓+temu的独享仓
# and yw.warehouse_name not like '%TK独享%')
sql = """
SELECT
    ps.sku sku, toString(toDate(toString(date_id))) date_id, yw.ebay_category_id AS category_id, yw.id AS warehouse_id,
    yw.warehouse_name AS warehouse_name, yw.warehouse_code AS warehouse_code, ywc.name AS warehouse,
    available_stock, purchase_on_way_count, allot_on_way_count AS on_way_stock, wait_outbound, frozen_stock,cargo_owner_id
FROM yb_datacenter.yb_stock AS ps
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse yw ON ps.warehouse_id = yw.id
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse_category ywc ON yw.ebay_category_id = ywc.id
WHERE 
    ps.date_id = '{0}'          -- 根据需要取时间
    and ps.cargo_owner_id = 8         -- 筛选货主ID为8的
    and (ps.available_stock > 0 or ps.allot_on_way_count > 0)
    and yw.warehouse_type in (2,3)    -- 筛选海外仓仓库
    and (yw.warehouse_other_type = 2 or yw.warehouse_name like '%独享%')  -- 筛选公告仓（非子仓）
    and yw.warehouse_name not in ('HC美国西仓','出口易美西仓-安大略仓','万邑通美南仓-USTX','亿迈CA01仓')  -- 剔除不常用仓库
    and ywc.name in ('美国仓','加拿大仓','墨西哥仓','澳洲仓','英国仓','德国仓','法国仓','俄罗斯仓','乌拉圭仓','西班牙仓','意大利仓')
ORDER BY date_id DESC
""".format(datetime.date.today().strftime('%Y%m%d'))
# and yw.warehouse_name not like '%TT%'
print(sql)

df_stock=get_data(sql,args=passowrd_c['121.37.30.78_z'],db_name='yb_datacenter')

#找出独享仓特有库存

df_stock=df_stock[['sku','warehouse_id','warehouse_name','warehouse_code','warehouse','available_stock','on_way_stock']]
df_stock=df_stock[df_stock['warehouse_name']!='谷仓美国东仓（14548独享）']
df_stock=df_stock[~df_stock['warehouse_name'].str.contains('TK')]
df_stock=df_stock[~df_stock['warehouse_name'].str.contains('15017')]
df_stock=df_stock[~df_stock['warehouse_name'].str.contains('14548')]  #14548 独享仓设置了共享   可以算公共库存

# 去掉VC项目独享仓
df_stock=df_stock[~df_stock['warehouse_name'].str.contains('VC')]


SELECT
    ps.sku sku, toString(toDate(toString(date_id))) date_id, yw.ebay_category_id AS category_id, yw.id AS warehouse_id,
    yw.warehouse_name AS warehouse_name, yw.warehouse_code AS warehouse_code, ywc.name AS warehouse,
    available_stock, purchase_on_way_count, allot_on_way_count AS on_way_stock, wait_outbound, frozen_stock,cargo_owner_id
FROM yb_datacenter.yb_stock AS ps
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse yw ON ps.warehouse_id = yw.id
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse_category ywc ON yw.ebay_category_id = ywc.id
WHERE 
    ps.date_id = '20250901'          -- 根据需要取时间
    and ps.cargo_owner_id = 8         -- 筛选货主ID为8的
    and (ps.available_stock > 0 or ps.allot_on_way_count > 0)
    and yw.warehouse_type in (2,3)    -- 筛选海外仓仓库
    and (yw.warehouse_other_type = 2 or yw.warehouse_name like '%独享%')  -- 筛选公告仓（非子仓）
    and yw.warehouse_name not in ('HC美国西仓','出口易美西仓-安大略仓','万邑通美南仓-USTX','亿迈CA01仓')  -- 剔除不常用仓库
    and ywc.name in ('美国仓','加拿大仓','墨西哥仓'

In [33]:
df_stock.head(1)

,sku,warehouse_id,warehouse_name,warehouse_code,warehouse,available_stock,on_way_stock
0,C11993RG,956,YM墨西哥仓,YM-MX-2,墨西哥仓,2,0


In [34]:
# #2024.11.26
# df_stock=df_stock[~df_stock['warehouse_name'].str.contains('1005')]
# df_stock=df_stock[~df_stock['warehouse_name'].str.contains('1006')]
# df_stock['站点']=df_stock['warehouse'].str.replace('仓','')
# df_stock.loc[df_stock['站点'].isin(['德国','法国','西班牙','意大利','荷兰','瑞典','波兰','土耳其','比利时']),'站点']='欧洲'
# df_stock.loc[df_stock['站点']=='澳洲','站点']='澳大利亚'
# df_stock['sku']=df_stock['sku'].astype(str).str.strip()
# print(df_stock[['warehouse','站点']].drop_duplicates())

# a=df_stock.groupby(['sku','站点']).agg({'available_stock':'sum','on_way_stock':'sum'}).reset_index()
# a=a.merge(path,on=['sku','站点'],how='left')
# a=a[a['责任人'].isnull()]
# a=a[a['站点'].isin(['加拿大', '欧洲', '澳大利亚', '美国', '英国'])]
# del a['责任人']
# a.to_excel(r'C:\Users\Administrator\Desktop\有库存但是未分配责任人的sku2025-06-11.xlsx',index=False)

In [35]:
# # 2025.1.2 临时新增

# df_stock['备注']=np.where(df_stock['warehouse_name'].str.contains('1005|1006'),'个人仓','公共仓')
# df_stock['站点']=df_stock['warehouse'].str.replace('仓','')
# df_stock.loc[df_stock['站点'].isin(['德国','法国','西班牙','意大利','荷兰','瑞典','波兰','土耳其','比利时']),'站点']='欧洲'
# df_stock.loc[df_stock['站点']=='澳洲','站点']='澳大利亚'
# df_stock['sku']=df_stock['sku'].astype(str).str.strip()
# print(df_stock[['warehouse','站点']].drop_duplicates())

# a=df_stock.groupby(['sku','站点','备注']).agg({'available_stock':'sum'}).unstack(fill_value=0).droplevel(axis=1,level=0).add_suffix('_可用库存').reset_index()
# a=a.merge(path,on=['sku','站点'],how='left')
# a=a[a['主体账号']=='控股公司项目：子龙']
# a.to_excel(r'C:\Users\Administrator\Desktop\责任sku库存2025-02-18.xlsx',index=False)

In [36]:
#独享仓对应使用人
# '谷仓美东toC代发仓(Temu独享)', '谷仓澳洲悉尼toC代发仓(Temu独享)', '谷仓捷克toC代发仓(Temu独享)', '谷仓英国toC代发仓(Temu独享)', '谷仓美西toC代发仓(Temu独享)'
# Temu 平台独享的 全公司公用的

warehouseid1=pd.read_excel(r"E:\20240712-Temu\海外仓虚拟仓-维护 20240914.xlsx",sheet_name='Sheet1',usecols=['id','虚拟仓','需求提报人'])
warehouseid1.rename(columns={'id':'warehouse_id'},inplace=True)
df_stock=df_stock.merge(warehouseid1,on=['warehouse_id'],how='left')

#确认是否有独享仓但是无使用人
print(set(df_stock[df_stock['warehouse_name'].str.contains('独享')&(df_stock['需求提报人'].isnull())]['warehouse_name']))
df_stock.loc[df_stock['需求提报人']=='刘子龙','需求提报人']='子龙'
df_stock=pd.concat([
df_stock[df_stock['需求提报人']!='李梦琴'].reindex(columns=['sku','warehouse','warehouse_name','available_stock','on_way_stock','需求提报人']),
df_stock[df_stock['需求提报人']=='李梦琴'].reindex(columns=['sku','warehouse','warehouse_name','available_stock','on_way_stock','需求提报人1'],fill_value='小雪糕').rename(columns={'需求提报人1':'需求提报人'}),
df_stock[df_stock['需求提报人']=='李梦琴'].reindex(columns=['sku','warehouse','warehouse_name','available_stock','on_way_stock','需求提报人1'],fill_value='毛竹').rename(columns={'需求提报人1':'需求提报人'})
])
print(set(df_stock['需求提报人']),set(path1['责任人']))
df_stock['站点']=df_stock['warehouse'].str.replace('仓','')

df_stock.loc[df_stock['站点'].isin(['德国','法国','西班牙','意大利','荷兰','瑞典','波兰','土耳其','比利时']),'站点']='欧洲'
df_stock.loc[df_stock['站点']=='澳洲','站点']='澳大利亚'
df_stock['sku']=df_stock['sku'].astype(str).str.strip()
print(df_stock[['warehouse','站点']].drop_duplicates())


{'亿迈NJ01仓（1006独享）', 'YM墨西哥toC代发仓(temu独享仓)', '出口易美东2仓（1006独享）', '出口易美西仓（1006独享）', '谷仓英国toC代发仓(Temu独享)', '出口易美东仓(1006独享)', '谷仓捷克toC代发仓(Temu独享)', '亿迈CA01仓（1006独享）'}
{nan, '子龙'} {'李茜', '李金强', 'libby', '杨静', '子龙'}
     warehouse    站点
0         墨西哥仓   墨西哥
3         加拿大仓   加拿大
6          英国仓    英国
17         澳洲仓  澳大利亚
40         德国仓    欧洲
47        俄罗斯仓   俄罗斯
71         美国仓    美国
1013       法国仓    欧洲
1253      西班牙仓    欧洲
3158      意大利仓    欧洲


In [37]:
df_stock.head(1)

,sku,warehouse,warehouse_name,available_stock,on_way_stock,需求提报人,站点
0,C11993RG,墨西哥仓,YM墨西哥仓,2,0,NaN,墨西哥


In [38]:
#2024.11.4 新增精铺sku库存

sql="""
select a.sku,a.available_stock,a.allot_on_way_count,b.name,b.country from jp_yb_stock_center.yb_stock a
left  join jp_yb_stock_center.yb_warehouse b
on a.warehouse_id=b.id
where (a.available_stock>0 or a.allot_on_way_count>0)
"""

df_stock_sass=sql_to_data(sql,db_name='jp_yb_stock_center',args=passowrd_m['121.37.228.71'])
df_stock_sass['站点']=df_stock_sass['country'].map({'AU':'澳大利亚','CA':'加拿大','CN':'中国','CZ':'欧洲', 'GB':'英国', 'US':'美国'})
df_stock_sass[['country','站点']].drop_duplicates().sort_values(by=['站点'])
del df_stock_sass['country']
df_stock_sass.rename(columns={'allot_on_way_count':'on_way_stock'},inplace=True)
df_stock_sass.rename(columns={'name':'warehouse_name'},inplace=True)

df_stock=pd.concat([df_stock,df_stock_sass])

In [39]:
df_stock_sass.head(1)

,sku,available_stock,on_way_stock,warehouse_name,站点
0,TS317BK01A-MB,36,0,AI萨凡纳2号toC代发仓,美国


In [40]:
df_stock.head(1)

,sku,warehouse,warehouse_name,available_stock,on_way_stock,需求提报人,站点
0,C11993RG,墨西哥仓,YM墨西哥仓,2,0,NaN,墨西哥


In [41]:
# 2025.1.2 新增独享仓在途库存 #虚拟仓在途库存数据不在云仓   需单独获取

sql="""
with
purchase_track as (
SELECT DISTINCT
sku ,is_busy,is_distribution,is_fumigation,is_drawback,destination_warehouse,
country,logistics_type,demand_number,purchase_number ,sales_name,cancel_qty,
purchase_order_status,order_quantity as purchase_qty
FROM yibai_plan_common_sync.yibai_purchase_pr_track_list
where purchase_type_id = 2
and suggest_status not in (3,4,5)
and purchase_number !=''
and push_time >= today()-300
and purchase_order_status not in (0,12,13,14,15)
),
purchase_order as (
select purchase_number,sku ,sum(instock_qty) instock_qty,sum(breakage_qty) breakage_qty
from yibai_purchase_sync.pur_warehouse_results
where quality_result =1 and instock_system in ('wms','(旧)WMS系统')
group by purchase_number ,sku ,purchase_qty
),
shipment_data as (
select sku,destination_warehouse,purchase_sn,shipment_status,
if(shipment_status=21,shipment_num,0) shipment_num,shelf_num
from yibai_plan_common_sync.yibai_oversea_shipment_list_detail a
left join (
select shipment_sn,shipment_status,shipment_type,b.name destination_warehouse
from yibai_plan_common_sync.yibai_oversea_shipment_list a
left join (select name,code from yb_datacenter.yb_warehouse) b on a.destination_warehouse_code=b.code
) b
on a.shipment_sn =b.shipment_sn
WHERE a.shipment_type in (1,2) and a.purchase_sn in (select DISTINCT purchase_number from purchase_track)
and b.shipment_status not in (3,14,15,23,24,26) and shipment_type in (1,2)
),
shipment_data_group as (
SELECT sku,purchase_sn, SUM(shipment_num) shipment_num,sum(shelf_num) shelf_num
from shipment_data group by sku,purchase_sn
)
select a.sku sku,if(a.sales_name = '','公司备货',a.sales_name) as sales_name,a.destination_warehouse warehouse_name,
a.purchase_number purchase_number,
a.purchase_qty purchase_qty,b.instock_qty instock_qty,b.breakage_qty breakage_qty,
if(c.shipment_num>instock_qty,instock_qty,c.shipment_num) shipment_num,c.shelf_num shelf_num,
a.purchase_qty as `采购数量`,
if(a.purchase_qty-b.instock_qty<0 or a.purchase_order_status in (8,9,11),0,a.purchase_qty-b.instock_qty) as `采购在途数量`,
if(b.instock_qty-b.breakage_qty-c.shipment_num<0,0,b.instock_qty-b.breakage_qty-c.shipment_num) as `中转仓库存数量`,
if(c.shipment_num-c.shelf_num<0,0,shipment_num-c.shelf_num) as `海外仓在途数量`
from purchase_track a
left join purchase_order b on a.sku=b.sku and a.purchase_number=b.purchase_number
left join shipment_data_group c on a.sku=c.sku and a.purchase_number=c.purchase_sn
where `海外仓在途数量` + `采购在途数量`+`中转仓库存数量` >0;
"""
df_stock_d=get_data(sql,args=passowrd_c['121.37.249.98'],db_name='yibai_plan_common_sync')
df_stock_d.rename(columns={'sales_name':'库存归属'},inplace=True)
df_stock_d['在途库存']=df_stock_d['采购在途数量']+df_stock_d['中转仓库存数量']+df_stock_d['海外仓在途数量']
df_stock_d.loc[df_stock_d['库存归属']=='刘子龙','库存归属']='子龙'
df_stock_d=df_stock_d[['sku','库存归属','warehouse_name','在途库存']]

sql="""
SELECT distinct yw.warehouse_name AS warehouse_name,ywc.name AS warehouse
from yibai_logistics_tms_sync.yibai_warehouse yw
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse_category ywc ON yw.ebay_category_id = ywc.id
"""

yibai_warehouse=get_data(sql,args=passowrd_c['121.37.30.78_z'],db_name='yibai_logistics_tms_sync')
df_stock_d=df_stock_d.merge(yibai_warehouse,on=['warehouse_name'],how='left')
df_stock_d.loc[df_stock_d['warehouse_name'].isin(['XY美东代发4仓', 'XY美东代发仓', 'XY美西代发6仓', 'XY美西代发仓']),'warehouse']='美国仓'


t=pd.read_csv(r"E:\20240712-Temu\销售名称对应仓库.csv")
t.rename(columns={'sales_name':'库存归属'},inplace=True)

df_stock_d=df_stock_d.merge(t[['库存归属','库存类型']],on=['库存归属'],how='left')
# df_stock_d[['warehouse_name','库存类型']].drop_duplicates().to_clipboard()
df_stock_d[['warehouse_name','库存类型']].drop_duplicates().to_clipboard()
df_stock_d=df_stock_d[df_stock_d['库存类型'].isin(['1005','1006',1005,1006,'Temu'])]
df_stock_d['站点']=df_stock_d['warehouse'].str.replace('仓','')
df_stock_d.loc[df_stock_d['站点'].isin(['德国','法国','西班牙','意大利','荷兰','瑞典','波兰','土耳其','比利时','捷克']),'站点']='欧洲'
df_stock_d.loc[df_stock_d['站点']=='澳洲','站点']='澳大利亚'
df_stock_d['sku']=df_stock_d['sku'].astype(str).str.strip()

df_stock_d1=df_stock_d.groupby(['sku','站点','库存类型']).agg({'在途库存':'sum'}).add_prefix('虚拟仓').reset_index()
df_stock_d1['需求提报人']=df_stock_d1['库存类型'].map({'1005':'彭柳森', '1006':'子龙', 'Temu':np.nan})
print(df_stock_d1[['库存类型','需求提报人']].drop_duplicates())


df_stock=pd.concat([
    df_stock[['sku','站点','需求提报人','available_stock','on_way_stock']],
    df_stock_d1[['sku','站点','需求提报人','虚拟仓在途库存']].rename(columns={'虚拟仓在途库存':'on_way_stock'})
])
df_stock['available_stock'].fillna(0,inplace=True)


    库存类型 需求提报人
0   Temu   NaN
1   1006    子龙
68  1005   彭柳森


In [42]:
df_stock_d1.head(1)

,sku,站点,库存类型,虚拟仓在途库存,需求提报人
0,07EGS46300,俄罗斯,Temu,30,NaN


In [43]:
df_stock.head(1)

,sku,站点,需求提报人,available_stock,on_way_stock
0,C11993RG,墨西哥,NaN,2.0,0


In [44]:
D=pd.DataFrame()


for i in set(path1['责任人']):
    print(i)
    d=df_stock.fillna(i).groupby(['sku','站点','需求提报人']).agg({'available_stock':'sum','on_way_stock':'sum'}).reset_index()
    d=d[d['需求提报人']==i]
    d.rename(columns={'需求提报人':'责任人'},inplace=True)
    D=pd.concat([D,d])
    

D['是否有可用库存']=np.where(D['available_stock']>0,'是','否')
D['是否可用库存>3']=np.where(D['available_stock']>3,'是','否')
path1=path1.merge(D,on=['sku','站点','责任人'],how='left')
path1['是否有可用库存'].fillna('否',inplace=True)
path1['是否可用库存>3'].fillna('否',inplace=True)

# path2=path1.drop(labels=['国内仓sku'],axis=1)
path2=path1.copy()
# 2024.10.15 后边统计易佰temu的责任sku刊登率的时候把插头剔除吧
path2=path2[~path2['sku'].isin(['SJ02606','SJ00970','SJ00969','SJ00968','SJ00967','SJ00966','SJ00965'])]
path2.rename(columns={'available_stock':'可用库存','on_way_stock':'在途库存'},inplace=True)
path2['是否已刊登']=path2['是否已刊登'].map({'是':1,'否':0})


#刊登覆盖率+加站率  分团队   可用库存>0 

for i in [1,2,3,5]:
    path2['刊登链接数是否超过{0}'.format(i)]=np.where(path2['刊登链接数']>=i,1,0)
    path2['加站链接数是否超过{0}'.format(i)]=np.where(path2['加站链接数']>=i,1,0)
    path2['核价通过链接数是否超过{0}'.format(i)]=np.where(path2['核价通过链接数']>=i,1,0)

李茜
李金强
libby
杨静
子龙


In [45]:
df_stock.head(1)

,sku,站点,需求提报人,available_stock,on_way_stock
0,C11993RG,墨西哥,NaN,2.0,0


In [46]:
D.head(1)

,sku,站点,责任人,available_stock,on_way_stock,是否有可用库存,是否可用库存>3
0,00EOT00708,加拿大,李茜,1.0,0,是,否


In [47]:
# 2025.5.9 新增海外仓低销且超180+库龄的SKU的加站率
# 海外仓低销：Temu平台sku+站点维度日销低于0.1[Temu全团队]
# 加权日均销量=近30天销量/30*0.1+近15天销量/15*0.2+近7天销量/7*0.7

# 动销率统计：  以加入站点时间为链接在售时间统计动销

sql_order="""
select b.platform_order_id,b.order_id as order_id,b.account_id as account_id,c.seller_sku as seller_sku,c.quantity,toDate(b.purchase_time)as purchase_time,
c.item_id,b.order_status,b.total_price,b.currency,b.ship_country,a.sku,a.quantity as sku_quantity
from
(select * from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id not like '%CJ%' and total_price !=0 and order_type !=4 and toDate(purchase_time)>='{0}')b 
left join (select * from yibai_oms_sync.yibai_oms_order_detail where order_id in (select order_id from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id not like '%CJ%' and total_price !=0 and order_type !=4 and toDate(purchase_time)>='{0}'))c on b.order_id=c.order_id
left join (select * from yibai_oms_sync.yibai_oms_order_sku where order_id in (select order_id from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id not like '%CJ%' and total_price !=0 and order_type !=4 and toDate(purchase_time)>='{0}'))a on a.order_id=c.order_id and a.order_detail_id=c.id

""".format((datetime.date.today()+datetime.timedelta(-30)).strftime('%Y-%m-%d'))
print(sql_order)

order=get_data(sql_order,args=passowrd_c['121.37.30.78'],db_name='yibai_oms_sync')
order.rename(columns={'b.platform_order_id':'platform_order_id','c.quantity':'quantity','c.item_id':'item_id','b.order_status':'order_status','b.total_price':'total_price','b.currency':'currency','b.ship_country':'ship_country','a.sku':'sku'},inplace=True)
order=order[order['order_status']!=80]
order['purchase_time']=pd.to_datetime(order['purchase_time'],format='%Y-%m-%d')


# 2024.11.11 新增sass系统订单

sql="""
select a.platform_order_id,a.order_id,c.short_name,b.seller_sku,b.quantity_old as quantity,date(a.purchase_time)as purchase_time,
b.item_id,a.order_status,b.total_price,a.currency,e.sku,e.quantity as sku_quantity,a.ship_country
from jp_saas_erp_order.dcm_order a
left join jp_saas_erp_order.dcm_order_detail b on a.order_id = b.order_id
left join jp_saas_erp_base.dcm_shop_v c on a.account_id=c.id  and a.shop_id=c.distributor_id
left join jp_saas_erp_order.dcm_order_sku e ON b.order_id = e.order_id AND b.id = e.order_detail_id
where a.platform_code='DMSTEMU'
and a.order_source !=4
and a.order_status !=40
and date(a.purchase_time)>='{0}'
""".format((datetime.date.today()+datetime.timedelta(-30)).strftime('%Y-%m-%d'))
print(sql)
order_sass=sql_to_data(sql,db_name='jp_saas_erp_order',args=passowrd_m['121.37.228.71'])
print(len(order_sass))
order_sass['short_name']=order_sass['short_name'].str.replace('-','')
order_sass=order_sass.merge(account[['账号简称','account_id','主体账号']].rename(columns={'账号简称':'short_name'}),on=['short_name'],how='left')
del order_sass['short_name']
print(set(order.columns)-set(order_sass.columns))
print(set(order_sass.columns)-set(order.columns))
order_sass['purchase_time']=pd.to_datetime(order_sass['purchase_time'],format='%Y-%m-%d')
order_sass=order_sass[order_sass['order_status']!=40]

order=pd.concat([order,order_sass.drop(labels=['主体账号'],axis=1)])
order['total_price']=order['total_price'].astype(float)
order['seller_sku']=order['seller_sku'].astype(str).str.strip()
order=order[order['purchase_time']<datetime.date.today().strftime('%Y-%m-%d')]


order=order.merge(listing_t1[['account_id','product_sku_id','站点']].rename(columns={'product_sku_id':'seller_sku'}).drop_duplicates(),on=['account_id','seller_sku'],how='left')
country={'us': '美国','it': '欧洲','sp': '欧洲','fr': '欧洲','de': '欧洲','uk': '英国','jp': '日本','ca': '加拿大','mx': '墨西哥','au': '澳大利亚','in': '印度','ae': '阿联酋','sg': '新加坡','nl': '欧洲','br': '巴西','sa': '沙特','se': '欧洲','pl': '欧洲','tr': '欧洲','be':'欧洲','es':'欧洲','gb':'英国','eu':'欧洲','sk':'欧洲','ie':'欧洲'}
order['站点'].fillna(order['ship_country'].str.lower().map(country),inplace=True)
order=order.merge(account[['account_id','主体账号']],on=['account_id'],how='left')


# 各团队加权销量=近30天销量/30*0.1+近15天销量/15*0.2+近7天销量/7*0.7
t30=order.groupby(['sku','站点']).agg({'sku_quantity':'sum'}).rename(columns={'sku_quantity':'近30天销量'})
t7=order[order['purchase_time']>=(datetime.date.today()+datetime.timedelta(-7)).strftime('%Y-%m-%d')].groupby(['sku','站点']).agg({'sku_quantity':'sum'}).rename(columns={'sku_quantity':'近7天销量'})
t15=order[order['purchase_time']>=(datetime.date.today()+datetime.timedelta(-15)).strftime('%Y-%m-%d')].groupby(['sku','站点']).agg({'sku_quantity':'sum'}).rename(columns={'sku_quantity':'近15天销量'})

quantity=t30.join(t15).join(t7).reset_index()
quantity['近15天销量'].fillna(0,inplace=True)
quantity['近7天销量'].fillna(0,inplace=True)
quantity['加权日均销量']=quantity['近7天销量']/7*0.7+quantity['近15天销量']/15*0.2+quantity['近30天销量']/30*0.1



select b.platform_order_id,b.order_id as order_id,b.account_id as account_id,c.seller_sku as seller_sku,c.quantity,toDate(b.purchase_time)as purchase_time,
c.item_id,b.order_status,b.total_price,b.currency,b.ship_country,a.sku,a.quantity as sku_quantity
from
(select * from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id not like '%CJ%' and total_price !=0 and order_type !=4 and toDate(purchase_time)>='2025-08-02')b 
left join (select * from yibai_oms_sync.yibai_oms_order_detail where order_id in (select order_id from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id not like '%CJ%' and total_price !=0 and order_type !=4 and toDate(purchase_time)>='2025-08-02'))c on b.order_id=c.order_id
left join (select * from yibai_oms_sync.yibai_oms_order_sku where order_id in (select order_id from yibai_oms_sync.yibai_oms_order where platform_code ='TEMU' and order_id not like '%RE%'and order_id n

In [48]:
#取sku库龄数据  确认是否存在精铺sku库龄数据？线下表，暂时不取

sql = '''
SELECT  
sku, charge_currency, cargo_type, ya.warehouse_code warehouse_code, yw.id as warehouse_id, 
yw.warehouse_name warehouse_name, ywc.name warehouse,
warehouse_stock, inventory_age, charge_total_price, 
case when inventory_age >= 30 then warehouse_stock else 0 end as age_30_plus,
case when inventory_age >= 60 then warehouse_stock else 0 end as age_60_plus,
case when inventory_age >= 90 then warehouse_stock else 0 end as age_90_plus,
case when inventory_age >= 120 then warehouse_stock else 0 end as age_120_plus,
case when inventory_age >= 150 then warehouse_stock else 0 end as age_150_plus,
case when inventory_age >= 180 then warehouse_stock else 0 end as age_180_plus,
case when inventory_age >= 210 then warehouse_stock else 0 end as age_210_plus,
case when inventory_age >= 270 then warehouse_stock else 0 end as age_270_plus,
case when inventory_age >= 360 then warehouse_stock else 0 end as age_360_plus
FROM yb_datacenter.yb_oversea_sku_age ya
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse AS yw ON ya.warehouse_code = yw.warehouse_code
LEFT JOIN yibai_logistics_tms_sync.yibai_warehouse_category AS ywc ON yw.ebay_category_id = ywc.id
WHERE date = formatDateTime(subtractDays(now(),2), '%Y-%m-%d') and status in (0,1)     
'''
# and ya.order_warehouse_code not like '%TT%'
# -- and yw.warehouse_name not like '%独享%' 
# -- 库龄表TT仓库code使用字段order_warehouse_code
df_stock=get_data(sql,args=passowrd_c['121.37.30.78_zz'],db_name='yb_datacenter')
df_stock['站点']=df_stock['warehouse'].str.replace('仓','')
df_stock.loc[df_stock['站点'].isin(['德国','法国','西班牙','意大利','荷兰','瑞典','波兰','土耳其','比利时']),'站点']='欧洲'
df_stock.loc[df_stock['站点']=='澳洲','站点']='澳大利亚'
df_stock['sku']=df_stock['sku'].astype(str).str.strip()
print(df_stock[['warehouse','站点']].drop_duplicates())
df_stock=df_stock.groupby(['sku','站点']).agg({'age_180_plus':'sum'}).reset_index()
df_stock.rename(columns={'age_180_plus':'超180+库龄库存'},inplace=True)
df_stock=df_stock.merge(quantity[['sku','站点','加权日均销量']],on=['sku','站点'],how='left')
df_stock['加权日均销量'].fillna(0,inplace=True)

# 2025.5.9 新增海外仓低销且超180+库龄的SKU的加站率
# 海外仓低销：Temu平台sku+站点维度日销低于0.1[Temu全团队]
# 加权日均销量=近30天销量/30*0.1+近15天销量/15*0.2+近7天销量/7*0.7
df_stock=df_stock[(df_stock['加权日均销量']<0.1)&(df_stock['超180+库龄库存']>0)]
df_stock['是否低销且存在180+库存']='是'

      warehouse    站点
0           美国仓    美国
2          加拿大仓   加拿大
3           澳洲仓  澳大利亚
4           英国仓    英国
13          德国仓    欧洲
153         日本仓    日本
163        None  None
521         法国仓    欧洲
10804      意大利仓    欧洲
11050      西班牙仓    欧洲


In [49]:
df_stock.head(1)

,sku,站点,超180+库龄库存,加权日均销量,是否低销且存在180+库存
2,04EGS52000,澳大利亚,1,0.0,是


In [50]:
path2=path2.merge(df_stock[['sku','站点','是否低销且存在180+库存']],on=['sku','站点'],how='left')
path2['是否低销且存在180+库存'].fillna('否',inplace=True)
path2.loc[(path2['是否有可用库存']=='否')&(path2['是否低销且存在180+库存']=='是'),'是否低销且存在180+库存']='否'
path2['是否低销且存在180+库存'].value_counts(dropna=False)

否    34752
是    11358
Name: 是否低销且存在180+库存, dtype: int64

In [51]:
path2.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架,是否已刊登,是否核价通过链接,是否已发布到站点,...,刊登链接数是否超过2,加站链接数是否超过2,核价通过链接数是否超过2,刊登链接数是否超过3,加站链接数是否超过3,核价通过链接数是否超过3,刊登链接数是否超过5,加站链接数是否超过5,核价通过链接数是否超过5,是否低销且存在180+库存
0,英国,GS21061,李金强,李金强,GS21061,是,是,1,1.0,1.0,...,1,1,1,1,1,1,1,1,1,否


In [52]:
path2['是否可上架'].value_counts()

是    37899
否     8211
Name: 是否可上架, dtype: int64

In [53]:
#不区分是否有可用库存  刊登覆盖率
t1=path2[(path2['是否可上架']=='是')].groupby(['主体账号']).agg({'是否责任类目':'count','是否已刊登':'sum','是否核价通过链接':'sum','是否已发布到站点':'sum','加站链接数是否超过2':'sum','加站链接数是否超过3':'sum'})
t1.rename(columns={'是否责任类目':'责任sku+目的国数量','是否已刊登':'已刊登责任sku+目的国数量','是否核价通过链接':'核价通过的责任sku+目的国数量','是否已发布到站点':'已加站责任sku+目的国数量','加站链接数是否超过2':'已加站2条责任sku+目的国数量','加站链接数是否超过3':'已加站3条责任sku+目的国数量'},inplace=True)
t1.loc['总计',:]=t1.sum()

t1.insert(2,'责任sku+目的国刊登率',t1['已刊登责任sku+目的国数量']/t1['责任sku+目的国数量'])
t1.insert(4,'责任sku+目的国核价通过率',t1['核价通过的责任sku+目的国数量']/t1['责任sku+目的国数量'])
t1.insert(6,'责任sku+目的国加站率',t1['已加站责任sku+目的国数量']/t1['责任sku+目的国数量'])
t1.insert(8,'责任sku+目的国2条加站率',t1['已加站2条责任sku+目的国数量']/t1['责任sku+目的国数量'])
t1.insert(10,'责任sku+目的国3条加站率',t1['已加站3条责任sku+目的国数量']/t1['责任sku+目的国数量'])
# 2025.1.10 新增
del t1['已刊登责任sku+目的国数量'],t1['责任sku+目的国刊登率'],t1['核价通过的责任sku+目的国数量'],t1['责任sku+目的国核价通过率']
t1.columns=pd.MultiIndex.from_arrays([['不区分是否有库存【已分配责任人的sku】']*len(t1.columns.tolist()),t1.columns.tolist()])


t2=path2[(path2['是否有可用库存']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号']).agg({'是否责任类目':'count','是否已刊登':'sum','是否核价通过链接':'sum','是否已发布到站点':'sum','加站链接数是否超过2':'sum','加站链接数是否超过3':'sum'})
t2.rename(columns={'是否责任类目':'责任sku+目的国数量','是否已刊登':'已刊登责任sku+目的国数量','是否核价通过链接':'核价通过的责任sku+目的国数量','是否已发布到站点':'已加站责任sku+目的国数量','加站链接数是否超过2':'已加站2条责任sku+目的国数量','加站链接数是否超过3':'已加站3条责任sku+目的国数量'},inplace=True)
t2.loc['总计',:]=t2.sum()
t2.insert(2,'责任sku+目的国刊登率',t2['已刊登责任sku+目的国数量']/t2['责任sku+目的国数量'])
t2.insert(4,'责任sku+目的国核价通过率',t2['核价通过的责任sku+目的国数量']/t2['责任sku+目的国数量'])
t2.insert(6,'责任sku+目的国加站率',t2['已加站责任sku+目的国数量']/t2['责任sku+目的国数量'])
t2.insert(8,'责任sku+目的国2条加站率',t2['已加站2条责任sku+目的国数量']/t2['责任sku+目的国数量'])
t2.insert(10,'责任sku+目的国3条加站率',t2['已加站3条责任sku+目的国数量']/t2['责任sku+目的国数量'])
# 2025.1.10 新增
del t2['已刊登责任sku+目的国数量'],t2['责任sku+目的国刊登率'],t2['核价通过的责任sku+目的国数量'],t2['责任sku+目的国核价通过率']
t2.columns=pd.MultiIndex.from_arrays([['在库>0']*len(t2.columns.tolist()),t2.columns.tolist()])

t3=path2[(path2['是否可用库存>3']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号']).agg({'是否责任类目':'count','是否已刊登':'sum','是否核价通过链接':'sum','是否已发布到站点':'sum','加站链接数是否超过2':'sum','加站链接数是否超过3':'sum'})
t3.rename(columns={'是否责任类目':'责任sku+目的国数量','是否已刊登':'已刊登责任sku+目的国数量','是否核价通过链接':'核价通过的责任sku+目的国数量','是否已发布到站点':'已加站责任sku+目的国数量','加站链接数是否超过2':'已加站2条责任sku+目的国数量','加站链接数是否超过3':'已加站3条责任sku+目的国数量'},inplace=True)
t3.loc['总计',:]=t3.sum()
t3.insert(2,'责任sku+目的国刊登率',t3['已刊登责任sku+目的国数量']/t3['责任sku+目的国数量'])
t3.insert(4,'责任sku+目的国核价通过率',t3['核价通过的责任sku+目的国数量']/t3['责任sku+目的国数量'])
t3.insert(6,'责任sku+目的国加站率',t3['已加站责任sku+目的国数量']/t3['责任sku+目的国数量'])
t3.insert(8,'责任sku+目的国2条加站率',t3['已加站2条责任sku+目的国数量']/t3['责任sku+目的国数量'])
t3.insert(10,'责任sku+目的国3条加站率',t3['已加站3条责任sku+目的国数量']/t3['责任sku+目的国数量'])
# 2025.1.10 新增
del t3['已刊登责任sku+目的国数量'],t3['责任sku+目的国刊登率'],t3['核价通过的责任sku+目的国数量'],t3['责任sku+目的国核价通过率']
t3.columns=pd.MultiIndex.from_arrays([['在库>3']*len(t3.columns.tolist()),t3.columns.tolist()])

#低销且存在180+库存刊登覆盖率
t4=path2[(path2['是否可上架']=='是')&(path2['是否低销且存在180+库存']=='是')].groupby(['主体账号']).agg({'是否责任类目':'count','是否已刊登':'sum','是否核价通过链接':'sum','是否已发布到站点':'sum','加站链接数是否超过2':'sum','加站链接数是否超过3':'sum'})
t4.rename(columns={'是否责任类目':'责任sku+目的国数量','是否已刊登':'已刊登责任sku+目的国数量','是否核价通过链接':'核价通过的责任sku+目的国数量','是否已发布到站点':'已加站责任sku+目的国数量','加站链接数是否超过2':'已加站2条责任sku+目的国数量','加站链接数是否超过3':'已加站3条责任sku+目的国数量'},inplace=True)
t4.loc['总计',:]=t4.sum()
t4.insert(2,'责任sku+目的国刊登率',t4['已刊登责任sku+目的国数量']/t4['责任sku+目的国数量'])
t4.insert(4,'责任sku+目的国核价通过率',t4['核价通过的责任sku+目的国数量']/t4['责任sku+目的国数量'])
t4.insert(6,'责任sku+目的国加站率',t4['已加站责任sku+目的国数量']/t4['责任sku+目的国数量'])
t4.insert(8,'责任sku+目的国2条加站率',t4['已加站2条责任sku+目的国数量']/t4['责任sku+目的国数量'])
t4.insert(10,'责任sku+目的国3条加站率',t4['已加站3条责任sku+目的国数量']/t4['责任sku+目的国数量'])

# 2025.1.10 新增
del t4['已刊登责任sku+目的国数量'],t4['责任sku+目的国刊登率'],t4['核价通过的责任sku+目的国数量'],t4['责任sku+目的国核价通过率']
t4.columns=pd.MultiIndex.from_arrays([['低销且存在180+库存[非精铺精品sku]']*len(t4.columns.tolist()),t4.columns.tolist()])


T=t1.join(t2).join(t3).join(t4)

In [54]:
T.tail(1)

不区分是否有库存【已分配责任人的sku】                                               \
              责任sku+目的国数量 已加站责任sku+目的国数量 责任sku+目的国加站率 已加站2条责任sku+目的国数量   
主体账号                                                                     
总计                37684.0        29082.0     0.771733          26165.0   

                                                           在库>0  \
     责任sku+目的国2条加站率 已加站3条责任sku+目的国数量 责任sku+目的国3条加站率 责任sku+目的国数量   
主体账号                                                              
总计         0.694327          23916.0       0.634646     13359.0   

                                  ...           在库>3                   \
     已加站责任sku+目的国数量 责任sku+目的国加站率  ... 责任sku+目的国2条加站率 已加站3条责任sku+目的国数量   
主体账号                              ...                                   
总计          10956.0     0.820121  ...       0.778697           7196.0   

                    低销且存在180+库存[非精铺精品sku]                              \
     责任sku+目的国3条加站率           责任sku+目的国数量 已加站责任sku+目的国数量 责任sku+目的国加站率   
主体账号                                                                    
总计         0.708407                8610.0         7562.0     0.878281   

                                                                      
     已加站2条责任sku+目的国数量 责任sku+目的国2条加站率 已加站3条责任sku+目的国数量 责任sku+目的国3条加站率  
主体账号                                                                  
总计             6721.0       0.780604           6051.0       0.702787  

[1 rows x 28 columns]

In [55]:
# #不区分是否有可用库存  刊登覆盖率
# t1=path2[(path2['是否可上架']=='是')].groupby(['主体账号','是否已刊登']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t1['责任sku+目的国数量']=t1['否']+t1['是']
# t1.rename(columns={'否':'未刊登sku数','是':'已刊登责任sku+目的国数量'},inplace=True)

# t2=path2[(path2['是否有可用库存']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','是否已刊登']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t2['库存>0 责任sku+目的国数量']=t2['否']+t2['是']
# t2.rename(columns={'否':'未刊登sku数(当前有库存sku)','是':'已刊登责任sku+目的国数量(库存>0 sku)'},inplace=True)

# t3=path2[(path2['是否可用库存>3']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','是否已刊登']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t3['库存>3 责任sku+目的国数量']=t3['否']+t3['是']
# t3.rename(columns={'否':'未刊登sku数(当前库存>3sku)','是':'已刊登责任sku+目的国数量(库存>3 sku)'},inplace=True)

# t=t1.join(t2).join(t3)
# t.insert(6,'库存>0 责任sku+目的国刊登率',t['已刊登责任sku+目的国数量(库存>0 sku)']/t['库存>0 责任sku+目的国数量'])
# t['辅助列']=[i.split('：')[0] for i in t.index]
# t.sort_values(by=['辅助列','库存>0 责任sku+目的国刊登率'],inplace=True)
# del t['辅助列']
# del t['库存>0 责任sku+目的国刊登率']
# t.loc['总计',:]=t.sum()
# t.insert(3,'责任sku+目的国刊登率',t['已刊登责任sku+目的国数量']/t['责任sku+目的国数量'])
# t.insert(7,'库存>0 责任sku+目的国刊登率',t['已刊登责任sku+目的国数量(库存>0 sku)']/t['库存>0 责任sku+目的国数量'])
# t.insert(11,'库存>3 责任sku+目的国刊登率',t['已刊登责任sku+目的国数量(库存>3 sku)']/t['库存>3 责任sku+目的国数量'])

In [56]:
# t.tail(1)

In [57]:
# #不区分是否有库存  加站覆盖率
# path2['是否核价通过链接1']=np.where(path2['是否核价通过链接']==1,'是','否')

# t1=path2[(path2['是否可上架']=='是')].groupby(['主体账号','是否核价通过链接1']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t1['责任sku+目的国数量']=t1['否']+t1['是']
# t1.rename(columns={'否':'未加站sku数','是':'已加站责任sku+目的国数量'},inplace=True)


# t2=path2[(path2['是否有可用库存']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','是否核价通过链接1']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t2['库存>0 责任sku+目的国数量']=t2['否']+t2['是']
# t2.rename(columns={'否':'未加站sku数(当前有库存sku)','是':'已加站责任sku+目的国数量(库存>0 sku)'},inplace=True)


# t3=path2[(path2['是否可用库存>3']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','是否核价通过链接1']).agg({'sku':'count'}).unstack(fill_value=0).droplevel(axis=1,level=0)
# t3['库存>3 责任sku+目的国数量']=t3['否']+t3['是']
# t3.rename(columns={'否':'未加站sku数(当前库存>3sku)','是':'已加站责任sku+目的国数量(库存>3 sku)'},inplace=True)

# T=t1.join(t2).join(t3)
# T.insert(6,'库存>0 责任sku+目的国加站覆盖率',T['已加站责任sku+目的国数量(库存>0 sku)']/T['库存>0 责任sku+目的国数量'])
# T['辅助列']=[i.split('：')[0] for i in T.index]
# T.sort_values(by=['辅助列','库存>0 责任sku+目的国加站覆盖率'],inplace=True)
# del T['辅助列']
# del T['库存>0 责任sku+目的国加站覆盖率']
# T.loc['总计',:]=T.sum()
# T.insert(3,'责任sku+目的国加站覆盖率',T['已加站责任sku+目的国数量']/T['责任sku+目的国数量'])
# T.insert(7,'库存>0 责任sku+目的国加站覆盖率',T['已加站责任sku+目的国数量(库存>0 sku)']/T['库存>0 责任sku+目的国数量'])
# T.insert(11,'库存>3 责任sku+目的国加站覆盖率',T['已加站责任sku+目的国数量(库存>3 sku)']/T['库存>3 责任sku+目的国数量'])

In [58]:
# T.tail(1)

In [59]:
path2.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架,是否已刊登,是否核价通过链接,是否已发布到站点,...,刊登链接数是否超过2,加站链接数是否超过2,核价通过链接数是否超过2,刊登链接数是否超过3,加站链接数是否超过3,核价通过链接数是否超过3,刊登链接数是否超过5,加站链接数是否超过5,核价通过链接数是否超过5,是否低销且存在180+库存
0,英国,GS21061,李金强,李金强,GS21061,是,是,1,1.0,1.0,...,1,1,1,1,1,1,1,1,1,否


In [60]:
T1=path2[(path2['是否有可用库存']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','站点']).agg({'sku':['count'],'刊登链接数是否超过1':['sum'],'刊登链接数是否超过2':['sum'],'刊登链接数是否超过3':['sum'],\
'刊登链接数是否超过5':['sum'],'核价通过链接数是否超过1':['sum'],'核价通过链接数是否超过2':['sum'],'核价通过链接数是否超过3':['sum'],'核价通过链接数是否超过5':['sum'],\
'加站链接数是否超过1':['sum'],'加站链接数是否超过2':['sum'],'加站链接数是否超过3':['sum'],'加站链接数是否超过5':['sum']})
T1.columns=[i+j for i,j in T1.columns]
T1.rename(columns={'skucount':'0SKU种数','刊登链接数是否超过1sum':'1条+刊登覆盖数','刊登链接数是否超过3sum':'3条+刊登覆盖数',
                   '刊登链接数是否超过5sum':'5条+刊登覆盖数','加站链接数是否超过1sum':'1条+加站覆盖数',
                   '加站链接数是否超过3sum':'3条+加站覆盖数','加站链接数是否超过5sum':'5条+加站覆盖数',
                   '核价通过链接数是否超过1sum':'1条+核价通过覆盖数','核价通过链接数是否超过3sum':'3条+核价通过覆盖数',
                   '核价通过链接数是否超过5sum':'5条+核价通过覆盖数','刊登链接数是否超过2sum':'2条+刊登覆盖数'
,'核价通过链接数是否超过2sum':'2条+核价通过覆盖数','加站链接数是否超过2sum':'2条+加站覆盖数'},inplace=True)
T1=T1.unstack().stack(level=0)
T1['sku+国家']=T1.sum(axis=1)
T1=T1.unstack().stack(level=0)

T1['1条+刊登覆盖率']=T1['1条+刊登覆盖数']/T1['0SKU种数']
T1['2条+刊登覆盖率']=T1['2条+刊登覆盖数']/T1['0SKU种数']
T1['3条+刊登覆盖率']=T1['3条+刊登覆盖数']/T1['0SKU种数']
T1['5条+刊登覆盖率']=T1['5条+刊登覆盖数']/T1['0SKU种数']
T1['1条+核价通过覆盖率']=T1['1条+核价通过覆盖数']/T1['0SKU种数']
T1['2条+核价通过覆盖率']=T1['2条+核价通过覆盖数']/T1['0SKU种数']
T1['3条+核价通过覆盖率']=T1['3条+核价通过覆盖数']/T1['0SKU种数']
T1['5条+核价通过覆盖率']=T1['5条+核价通过覆盖数']/T1['0SKU种数']
T1['1条+加站覆盖率']=T1['1条+加站覆盖数']/T1['0SKU种数']
T1['2条+加站覆盖率']=T1['2条+加站覆盖数']/T1['0SKU种数']
T1['3条+加站覆盖率']=T1['3条+加站覆盖数']/T1['0SKU种数']
T1['5条+加站覆盖率']=T1['5条+加站覆盖数']/T1['0SKU种数']


del T1['1条+刊登覆盖数'],T1['1条+加站覆盖数'],T1['1条+核价通过覆盖数'],T1['3条+刊登覆盖数'],T1['3条+加站覆盖数'],T1['3条+核价通过覆盖数'],T1['5条+刊登覆盖数'],T1['5条+加站覆盖数'],T1['5条+核价通过覆盖数'],T1['2条+刊登覆盖数'],T1['2条+加站覆盖数'],T1['2条+核价通过覆盖数']
T1=T1.unstack(fill_value=0).stack(level=0)
T1['辅助列']=[i[-5:] for _,i in T1.index]
T1['辅助列1']=[i for i,_ in T1.index]
T1['辅助列']=T1['辅助列'].map({'SKU种数':0,'刊登覆盖率':1,'通过覆盖率':2,'加站覆盖率':3})
T1.sort_values(by=['辅助列1','辅助列'],inplace=True)
del T1['辅助列'],T1['辅助列1']
T1.rename(index={'0SKU种数':'SKU种数'},inplace=True)


In [61]:
T1.tail(15)

站点                   sku+国家       加拿大         欧洲       澳大利亚         美国  \
主体账号                                                                     
李金强  3条+加站覆盖率      0.712740  0.735370   0.754637   0.555891   0.747228   
     5条+加站覆盖率      0.617167  0.607803   0.686341   0.507553   0.643016   
杨静   SKU种数       212.000000  4.000000  61.000000  39.000000  64.000000   
     1条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     2条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     3条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     5条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     1条+核价通过覆盖率    0.971698  1.000000   0.967213   1.000000   0.968750   
     2条+核价通过覆盖率    0.971698  1.000000   0.967213   1.000000   0.968750   
     3条+核价通过覆盖率    0.952830  1.000000   0.967213   0.974359   0.953125   
     5条+核价通过覆盖率    0.910377  0.750000   0.934426   0.923077   0.890625   
     1条+加站覆盖率      0.957547  1.000000   0.967213   0.974359   0.953125   
     2条+加站覆盖率      0.948113  1.000000   0.950820   0.948718   0.953125   
     3条+加站覆盖率      0.891509  1.000000   0.934426   0.897436   0.828125   
     5条+加站覆盖率      0.754717  0.750000   0.868852   0.846154   0.609375   

站点                      英国  
主体账号                        
李金强  3条+加站覆盖率     0.643175  
     5条+加站覆盖率     0.568254  
杨静   SKU种数       44.000000  
     1条+刊登覆盖率     1.000000  
     2条+刊登覆盖率     1.000000  
     3条+刊登覆盖率     1.000000  
     5条+刊登覆盖率     1.000000  
     1条+核价通过覆盖率   0.954545  
     2条+核价通过覆盖率   0.954545  
     3条+核价通过覆盖率   0.909091  
     5条+核价通过覆盖率   0.909091  
     1条+加站覆盖率     0.931818  
     2条+加站覆盖率     0.931818  
     3条+加站覆盖率     0.909091  
     5条+加站覆盖率     0.727273

In [62]:
path2.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架,是否已刊登,是否核价通过链接,是否已发布到站点,...,刊登链接数是否超过2,加站链接数是否超过2,核价通过链接数是否超过2,刊登链接数是否超过3,加站链接数是否超过3,核价通过链接数是否超过3,刊登链接数是否超过5,加站链接数是否超过5,核价通过链接数是否超过5,是否低销且存在180+库存
0,英国,GS21061,李金强,李金强,GS21061,是,是,1,1.0,1.0,...,1,1,1,1,1,1,1,1,1,否


In [63]:
#刊登覆盖率+加站率  分团队   可用库存>3

T2=path2[(path2['是否可用库存>3']=='是')&(path2['是否可上架']=='是')].groupby(['主体账号','站点']).agg({'sku':['count'],'刊登链接数是否超过1':['sum'],'刊登链接数是否超过2':['sum'],'刊登链接数是否超过3':['sum'],\
'刊登链接数是否超过5':['sum'],'核价通过链接数是否超过1':['sum'],'核价通过链接数是否超过2':['sum'],'核价通过链接数是否超过3':['sum'],'核价通过链接数是否超过5':['sum'],\
'加站链接数是否超过1':['sum'],'加站链接数是否超过2':['sum'],'加站链接数是否超过3':['sum'],'加站链接数是否超过5':['sum']})
T2.columns=[i+j for i,j in T2.columns]
T2.rename(columns={'skucount':'0SKU种数','刊登链接数是否超过1sum':'1条+刊登覆盖数','刊登链接数是否超过2sum':'2条+刊登覆盖数','刊登链接数是否超过3sum':'3条+刊登覆盖数',
                   '刊登链接数是否超过5sum':'5条+刊登覆盖数','加站链接数是否超过1sum':'1条+加站覆盖数','加站链接数是否超过2sum':'2条+加站覆盖数',
                   '加站链接数是否超过3sum':'3条+加站覆盖数','加站链接数是否超过5sum':'5条+加站覆盖数',
                  '核价通过链接数是否超过1sum':'1条+核价通过覆盖数','核价通过链接数是否超过2sum':'2条+核价通过覆盖数','核价通过链接数是否超过3sum':'3条+核价通过覆盖数',
                   '核价通过链接数是否超过5sum':'5条+核价通过覆盖数'},inplace=True)
T2=T2.unstack().stack(level=0)
T2['sku+国家']=T2.sum(axis=1)
T2=T2.unstack().stack(level=0)
T2['1条+刊登覆盖率']=T2['1条+刊登覆盖数']/T2['0SKU种数']
T2['2条+刊登覆盖率']=T2['2条+刊登覆盖数']/T2['0SKU种数']
T2['3条+刊登覆盖率']=T2['3条+刊登覆盖数']/T2['0SKU种数']
T2['5条+刊登覆盖率']=T2['5条+刊登覆盖数']/T2['0SKU种数']
T2['1条+核价通过覆盖率']=T2['1条+核价通过覆盖数']/T2['0SKU种数']
T2['2条+核价通过覆盖率']=T2['2条+核价通过覆盖数']/T2['0SKU种数']
T2['3条+核价通过覆盖率']=T2['3条+核价通过覆盖数']/T2['0SKU种数']
T2['5条+核价通过覆盖率']=T2['5条+核价通过覆盖数']/T2['0SKU种数']
T2['1条+加站覆盖率']=T2['1条+加站覆盖数']/T2['0SKU种数']
T2['2条+加站覆盖率']=T2['2条+加站覆盖数']/T2['0SKU种数']
T2['3条+加站覆盖率']=T2['3条+加站覆盖数']/T2['0SKU种数']
T2['5条+加站覆盖率']=T2['5条+加站覆盖数']/T2['0SKU种数']
del T2['1条+刊登覆盖数'],T2['1条+加站覆盖数'],T2['1条+核价通过覆盖数'],T2['3条+刊登覆盖数'],T2['3条+加站覆盖数'],T2['3条+核价通过覆盖数'],T2['5条+刊登覆盖数'],T2['5条+加站覆盖数'],T2['5条+核价通过覆盖数'],T2['2条+刊登覆盖数'],T2['2条+加站覆盖数'],T2['2条+核价通过覆盖数']
T2=T2.unstack(fill_value=0).stack(level=0)
T2['辅助列']=[i[-5:] for _,i in T2.index]
T2['辅助列1']=[i for i,_ in T2.index]
T2['辅助列']=T2['辅助列'].map({'SKU种数':0,'刊登覆盖率':1,'通过覆盖率':2,'加站覆盖率':3})
T2.sort_values(by=['辅助列1','辅助列'],inplace=True)
del T2['辅助列'],T2['辅助列1']
T2.rename(index={'0SKU种数':'SKU种数'},inplace=True)

In [64]:
T2.tail(15)

站点                   sku+国家       加拿大         欧洲       澳大利亚         美国  \
主体账号                                                                     
李金强  3条+加站覆盖率      0.777545  0.772807   0.784694   0.808824   0.802372   
     5条+加站覆盖率      0.673889  0.639181   0.712736   0.754202   0.700922   
杨静   SKU种数       173.000000  4.000000  56.000000  26.000000  49.000000   
     1条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     2条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     3条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     5条+刊登覆盖率      1.000000  1.000000   1.000000   1.000000   1.000000   
     1条+核价通过覆盖率    0.971098  1.000000   0.964286   1.000000   0.979592   
     2条+核价通过覆盖率    0.971098  1.000000   0.964286   1.000000   0.979592   
     3条+核价通过覆盖率    0.965318  1.000000   0.964286   1.000000   0.979592   
     5条+核价通过覆盖率    0.930636  0.750000   0.928571   0.961538   0.938776   
     1条+加站覆盖率      0.971098  1.000000   0.964286   1.000000   0.979592   
     2条+加站覆盖率      0.965318  1.000000   0.946429   1.000000   0.979592   
     3条+加站覆盖率      0.924855  1.000000   0.928571   0.923077   0.918367   
     5条+加站覆盖率      0.791908  0.750000   0.857143   0.923077   0.673469   

站点                      英国  
主体账号                        
李金强  3条+加站覆盖率     0.726514  
     5条+加站覆盖率     0.644050  
杨静   SKU种数       38.000000  
     1条+刊登覆盖率     1.000000  
     2条+刊登覆盖率     1.000000  
     3条+刊登覆盖率     1.000000  
     5条+刊登覆盖率     1.000000  
     1条+核价通过覆盖率   0.947368  
     2条+核价通过覆盖率   0.947368  
     3条+核价通过覆盖率   0.921053  
     5条+核价通过覆盖率   0.921053  
     1条+加站覆盖率     0.947368  
     2条+加站覆盖率     0.947368  
     3条+加站覆盖率     0.921053  
     5条+加站覆盖率     0.763158

In [65]:
path2.head(1)

,站点,sku,责任人,主体账号,国内仓sku,是否责任类目,是否可上架,是否已刊登,是否核价通过链接,是否已发布到站点,...,刊登链接数是否超过2,加站链接数是否超过2,核价通过链接数是否超过2,刊登链接数是否超过3,加站链接数是否超过3,核价通过链接数是否超过3,刊登链接数是否超过5,加站链接数是否超过5,核价通过链接数是否超过5,是否低销且存在180+库存
0,英国,GS21061,李金强,李金强,GS21061,是,是,1,1.0,1.0,...,1,1,1,1,1,1,1,1,1,否


In [66]:
sku.head(1)

,国内仓sku,站点,主体账号,是否核价通过链接,是否已发布到站点,核价通过链接数,加站链接数,刊登链接数,是否已刊登
0,\n2717250013512,美国,刘子龙,1,0,1,0,1,是


In [67]:
#团队汇总  看全团队的刊登情况  不区分是否责任sku
sku1=sku.groupby(['国内仓sku','站点']).agg({'是否核价通过链接':'max','是否已发布到站点':'max','加站链接数':'sum','刊登链接数':'sum','核价通过链接数':'sum'}).reset_index()
sku1['是否已刊登']='是'

# 2025.7.22(path2['是否可上架']=='是')
Path2=path2[['sku','站点','国内仓sku','可用库存','是否有可用库存','是否可用库存>3','是否可上架']].merge(sku1,on=['国内仓sku','站点'],how='left')
print(len(Path2)-len(path2))
Path2['是否核价通过链接'].fillna(0,inplace=True)
Path2['是否已发布到站点'].fillna(0,inplace=True)
Path2['加站链接数'].fillna(0,inplace=True)
Path2['刊登链接数'].fillna(0,inplace=True)
Path2['核价通过链接数'].fillna(0,inplace=True)
Path2['是否已刊登'].fillna('否',inplace=True)


0


In [68]:
sku1.head(1)

,国内仓sku,站点,是否核价通过链接,是否已发布到站点,加站链接数,刊登链接数,核价通过链接数,是否已刊登
0,\n2717250013512,美国,1,0,0,1,1,是


In [69]:
Path2.head(1)

,sku,站点,国内仓sku,可用库存,是否有可用库存,是否可用库存>3,是否可上架,是否核价通过链接,是否已发布到站点,加站链接数,刊登链接数,核价通过链接数,是否已刊登
0,GS21061,英国,GS21061,NaN,否,否,是,1.0,1.0,32.0,100.0,63.0,是


In [70]:
#刊登覆盖率+加站率  不分团队   可用库存>0   不区分是否责任人刊登

for i in [1,2,3,5]:
    Path2['刊登链接数是否超过{0}'.format(i)]=np.where(Path2['刊登链接数']>=i,1,0)
    Path2['加站链接数是否超过{0}'.format(i)]=np.where(Path2['加站链接数']>=i,1,0)
    Path2['核价通过链接数是否超过{0}'.format(i)]=np.where(Path2['核价通过链接数']>=i,1,0)

T11=Path2[(Path2['是否有可用库存']=='是')&(Path2['是否可上架']=='是')].groupby(['站点']).agg({'sku':['count'],'刊登链接数是否超过1':['sum'],'刊登链接数是否超过2':['sum'],'刊登链接数是否超过3':['sum'],\
'刊登链接数是否超过5':['sum'],'核价通过链接数是否超过1':['sum'],'核价通过链接数是否超过2':['sum'],'核价通过链接数是否超过3':['sum'],'核价通过链接数是否超过5':['sum'],\
'加站链接数是否超过1':['sum'],'加站链接数是否超过2':['sum'],'加站链接数是否超过3':['sum'],'加站链接数是否超过5':['sum']})
T11.columns=[i+j for i,j in T11.columns]
T11.rename(columns={'skucount':'0SKU种数','刊登链接数是否超过1sum':'1条+刊登覆盖数','刊登链接数是否超过2sum':'2条+刊登覆盖数','刊登链接数是否超过3sum':'3条+刊登覆盖数',
                   '刊登链接数是否超过5sum':'5条+刊登覆盖数','加站链接数是否超过1sum':'1条+加站覆盖数','加站链接数是否超过2sum':'2条+加站覆盖数',
                   '加站链接数是否超过3sum':'3条+加站覆盖数','加站链接数是否超过5sum':'5条+加站覆盖数',
                   '核价通过链接数是否超过1sum':'1条+核价通过覆盖数','核价通过链接数是否超过2sum':'2条+核价通过覆盖数','核价通过链接数是否超过3sum':'3条+核价通过覆盖数',
                   '核价通过链接数是否超过5sum':'5条+核价通过覆盖数'},inplace=True)
T11=T11.T
T11['sku+国家']=T11.sum(axis=1)
T11=T11.T
T11['1条+刊登覆盖率']=T11['1条+刊登覆盖数']/T11['0SKU种数']
T11['2条+刊登覆盖率']=T11['2条+刊登覆盖数']/T11['0SKU种数']
T11['3条+刊登覆盖率']=T11['3条+刊登覆盖数']/T11['0SKU种数']
T11['5条+刊登覆盖率']=T11['5条+刊登覆盖数']/T11['0SKU种数']
T11['1条+核价通过覆盖率']=T11['1条+核价通过覆盖数']/T11['0SKU种数']
T11['2条+核价通过覆盖率']=T11['2条+核价通过覆盖数']/T11['0SKU种数']
T11['3条+核价通过覆盖率']=T11['3条+核价通过覆盖数']/T11['0SKU种数']
T11['5条+核价通过覆盖率']=T11['5条+核价通过覆盖数']/T11['0SKU种数']
T11['1条+加站覆盖率']=T11['1条+加站覆盖数']/T11['0SKU种数']
T11['2条+加站覆盖率']=T11['2条+加站覆盖数']/T11['0SKU种数']
T11['3条+加站覆盖率']=T11['3条+加站覆盖数']/T11['0SKU种数']
T11['5条+加站覆盖率']=T11['5条+加站覆盖数']/T11['0SKU种数']
del T11['1条+刊登覆盖数'],T11['1条+加站覆盖数'],T11['1条+核价通过覆盖数'],T11['3条+刊登覆盖数'],T11['3条+加站覆盖数'],T11['3条+核价通过覆盖数'],T11['5条+刊登覆盖数'],T11['5条+加站覆盖数'],T11['5条+核价通过覆盖数'],T11['2条+刊登覆盖数'],T11['2条+加站覆盖数'],T11['2条+核价通过覆盖数']
T11=T11.T
T11.rename(index={'0SKU种数':'SKU种数'},inplace=True)
T11['主体账号']='Temu全团队'
T11=T11.set_index('主体账号',append=True).swaplevel()


In [71]:
T11

站点                          加拿大           欧洲         澳大利亚           美国  \
主体账号                                                                     
Temu全团队 SKU种数       4485.000000  3048.000000  1121.000000  2977.000000   
        1条+刊登覆盖率       0.935117     0.960630     0.729706     0.963722   
        2条+刊登覆盖率       0.921516     0.948163     0.706512     0.950621   
        3条+刊登覆盖率       0.907023     0.935696     0.680642     0.941888   
        5条+刊登覆盖率       0.878484     0.911417     0.663693     0.917366   
        1条+核价通过覆盖率     0.923746     0.928806     0.710972     0.941888   
        2条+核价通过覆盖率     0.890524     0.896325     0.677074     0.899899   
        3条+核价通过覆盖率     0.854181     0.869423     0.647636     0.861606   
        5条+核价通过覆盖率     0.769677     0.807415     0.618198     0.781323   
        1条+加站覆盖率       0.890524     0.877953     0.683318     0.887807   
        2条+加站覆盖率       0.826310     0.824475     0.634255     0.819617   
        3条+加站覆盖率       0.762988     0.785433     0.599465     0.760833   
        5条+加站覆盖率       0.643255     0.720472     0.553970     0.659053   

站点                           英国        sku+国家  
主体账号                                           
Temu全团队 SKU种数       1801.000000  13432.000000  
        1条+刊登覆盖率       0.815658      0.914086  
        2条+刊登覆盖率       0.796780      0.899345  
        3条+刊登覆盖率       0.786230      0.886167  
        5条+刊登覆盖率       0.770128      0.862120  
        1条+核价通过覆盖率     0.794003      0.893761  
        2条+核价通过覆盖率     0.769572      0.859887  
        3条+核价通过覆盖率     0.750139      0.828097  
        5条+核价通过覆盖率     0.709606      0.760125  
        1条+加站覆盖率       0.747918      0.850655  
        2条+加站覆盖率       0.714048      0.793329  
        3条+加站覆盖率       0.682954      0.743225  
        5条+加站覆盖率       0.615769      0.653142

In [72]:
#刊登覆盖率+加站率  不分团队   可用库存>3   不区分是否责任人刊登

T12=Path2[(Path2['是否可用库存>3']=='是')&(Path2['是否可上架']=='是')].groupby(['站点']).agg({'sku':['count'],'刊登链接数是否超过1':['sum'],'刊登链接数是否超过2':['sum'],'刊登链接数是否超过3':['sum'],\
'刊登链接数是否超过5':['sum'],'核价通过链接数是否超过1':['sum'],'核价通过链接数是否超过2':['sum'],'核价通过链接数是否超过3':['sum'],'核价通过链接数是否超过5':['sum'],\
'加站链接数是否超过1':['sum'],'加站链接数是否超过2':['sum'],'加站链接数是否超过3':['sum'],'加站链接数是否超过5':['sum']})
T12.columns=[i+j for i,j in T12.columns]
T12.rename(columns={'skucount':'0SKU种数','刊登链接数是否超过1sum':'1条+刊登覆盖数','刊登链接数是否超过2sum':'2条+刊登覆盖数','刊登链接数是否超过3sum':'3条+刊登覆盖数',
                   '刊登链接数是否超过5sum':'5条+刊登覆盖数','加站链接数是否超过1sum':'1条+加站覆盖数','加站链接数是否超过2sum':'2条+加站覆盖数',
                   '加站链接数是否超过3sum':'3条+加站覆盖数','加站链接数是否超过5sum':'5条+加站覆盖数',
                   '核价通过链接数是否超过1sum':'1条+核价通过覆盖数','核价通过链接数是否超过2sum':'2条+核价通过覆盖数' ,'核价通过链接数是否超过3sum':'3条+核价通过覆盖数',
                   '核价通过链接数是否超过5sum':'5条+核价通过覆盖数'},inplace=True)
T12=T12.T
T12['sku+国家']=T12.sum(axis=1)
T12=T12.T
T12['1条+刊登覆盖率']=T12['1条+刊登覆盖数']/T12['0SKU种数']
T12['2条+刊登覆盖率']=T12['2条+刊登覆盖数']/T12['0SKU种数']
T12['3条+刊登覆盖率']=T12['3条+刊登覆盖数']/T12['0SKU种数']
T12['5条+刊登覆盖率']=T12['5条+刊登覆盖数']/T12['0SKU种数']
T12['1条+核价通过覆盖率']=T12['1条+核价通过覆盖数']/T12['0SKU种数']
T12['2条+核价通过覆盖率']=T12['2条+核价通过覆盖数']/T12['0SKU种数']
T12['3条+核价通过覆盖率']=T12['3条+核价通过覆盖数']/T12['0SKU种数']
T12['5条+核价通过覆盖率']=T12['5条+核价通过覆盖数']/T12['0SKU种数']
T12['1条+加站覆盖率']=T12['1条+加站覆盖数']/T12['0SKU种数']
T12['2条+加站覆盖率']=T12['2条+加站覆盖数']/T12['0SKU种数']
T12['3条+加站覆盖率']=T12['3条+加站覆盖数']/T12['0SKU种数']
T12['5条+加站覆盖率']=T12['5条+加站覆盖数']/T12['0SKU种数']
del T12['1条+刊登覆盖数'],T12['1条+加站覆盖数'],T12['1条+核价通过覆盖数'],T12['3条+刊登覆盖数'],T12['3条+加站覆盖数'],T12['3条+核价通过覆盖数'],T12['5条+刊登覆盖数'],T12['5条+加站覆盖数'],T12['5条+核价通过覆盖数'],T12['2条+刊登覆盖数'],T12['2条+加站覆盖数'],T12['2条+核价通过覆盖数']
T12=T12.T
T12.rename(index={'0SKU种数':'SKU种数'},inplace=True)
T12['主体账号']='Temu全团队'
T12=T12.set_index('主体账号',append=True).swaplevel()


In [73]:
T12

站点                          加拿大           欧洲        澳大利亚           美国  \
主体账号                                                                    
Temu全团队 SKU种数       3997.000000  2359.000000  579.000000  2148.000000   
        1条+刊登覆盖率       0.974731     0.980076    0.949914     0.963222   
        2条+刊登覆盖率       0.961221     0.969055    0.941278     0.954842   
        3条+刊登覆盖率       0.946460     0.956761    0.918826     0.946462   
        5条+刊登覆盖率       0.916938     0.932599    0.903282     0.925047   
        1条+核价通过覆盖率     0.963473     0.947435    0.934370     0.947858   
        2条+核价通过覆盖率     0.928697     0.914370    0.910190     0.912477   
        3条+核价通过覆盖率     0.892920     0.889360    0.886010     0.884078   
        5条+核价通过覆盖率     0.804353     0.823230    0.858377     0.805400   
        1条+加站覆盖率       0.929947     0.903773    0.908463     0.911546   
        2条+加站覆盖率       0.861646     0.846121    0.865285     0.849628   
        3条+加站覆盖率       0.797348     0.808393    0.829016     0.797020   
        5条+加站覆盖率       0.673255     0.739720    0.780656     0.694600   

站点                           英国        sku+国家  
主体账号                                           
Temu全团队 SKU种数       1142.000000  10225.000000  
        1条+刊登覆盖率       0.901051      0.963912  
        2条+刊登覆盖率       0.880911      0.951589  
        3条+刊登覆盖率       0.869527      0.938680  
        5条+刊登覆盖率       0.852890      0.914328  
        1条+核价通过覆盖率     0.878284      0.945330  
        2条+核价通过覆盖率     0.850263      0.912176  
        3条+核价通过覆盖率     0.828371      0.882641  
        5条+核价通过覆盖率     0.787215      0.810073  
        1条+加站覆盖率       0.834501      0.908166  
        2条+加站覆盖率       0.795972      0.848411  
        3条+加站覆盖率       0.761821      0.797653  
        5条+加站覆盖率       0.691769      0.701222

In [74]:
text1="""
加站率  统计时间：{0}
""".format(datetime.date.today().strftime('%Y/%m/%d'))


text3="""统计时间：{0} 可用库存>0 【Temu全团队：分配责任人的sku+目的国在任意主体账号刊登情况，其他为对应责任人刊登情况】""".format(datetime.date.today().strftime('%Y/%m/%d'))

text4="""统计时间：{0} 可用库存>3 【Temu全团队：分配责任人的sku+目的国在任意主体账号刊登情况，其他为对应责任人刊登情况】""".format(datetime.date.today().strftime('%Y/%m/%d'))
text5="""
PS：
1、排除提报TEMU不上架产品以及插头sku
2、总计按sku+目的国分配责任人的sku在对应责任人刊登情况
3、海外仓低销：全平台日均销量低于0.1[加权日均销量=近30天销量/30*0.1+近15天销量/15*0.2+近7天销量/7*0.7]
"""

writer=pd.ExcelWriter(r'E:\20240712-Temu\责任类目刊登情况统计{0}.xlsx'.format(datetime.date.today().strftime('%Y-%m-%d')))

pd.DataFrame([text1]).to_excel(writer,sheet_name='数据统计',startrow=0,index=False,header=False)
T.to_excel(writer,sheet_name='数据统计',startrow=1)
pd.DataFrame([text5]).to_excel(writer,sheet_name='数据统计',startrow=len(T)+5,index=False,header=False)


# 汇总   可用库存>0 
T111=pd.concat([T11,T1])
T111=T111[['欧洲','英国','美国','加拿大','澳大利亚','sku+国家']]
T111.rename(columns={'加拿大':'CA','欧洲':'EUR','澳大利亚':'AU','美国':'US','英国':'UK'},inplace=True)

# 2025.1.10新增
columns1=set([i for i,_ in T111.index])
columns2=['1条+刊登覆盖率',
 '1条+核价通过覆盖率',
'2条+刊登覆盖率',
'2条+核价通过覆盖率',
 '3条+刊登覆盖率',
 '3条+核价通过覆盖率',
 '5条+刊登覆盖率',
 '5条+核价通过覆盖率',]
T111.drop(labels=[i for i in product(columns1,columns2)],axis=0,inplace=True)



# 汇总 可用库存>3  
T122=pd.concat([T12,T2])
T122=T122[['欧洲','英国','美国','加拿大','澳大利亚','sku+国家']]
T122.rename(columns={'加拿大':'CA','欧洲':'EUR','澳大利亚':'AU','美国':'US','英国':'UK'},inplace=True)


# 2025.1.10新增
columns1=set([i for i,_ in T122.index])
columns2=['1条+刊登覆盖率',
 '1条+核价通过覆盖率',
'2条+刊登覆盖率',
'2条+核价通过覆盖率',
 '3条+刊登覆盖率',
 '3条+核价通过覆盖率',
 '5条+刊登覆盖率',
 '5条+核价通过覆盖率',]
T122.drop(labels=[i for i in product(columns1,columns2)],axis=0,inplace=True)


pd.DataFrame([text3]).to_excel(writer,sheet_name='数据统计1',startrow=0,startcol=0,index=False,header=False)
T111.to_excel(writer,sheet_name='数据统计1',startrow=1,startcol=0)


pd.DataFrame([text4]).to_excel(writer,sheet_name='数据统计1',startrow=0,startcol=T111.shape[1]+5,index=False,header=False)
T122.to_excel(writer,sheet_name='数据统计1',startrow=1,startcol=T111.shape[1]+5)


#责任人刊登情况明细
path2.to_excel(writer,sheet_name='责任人刊登情况数据明细',index=False) 
#不区分责任人明细
Path2.to_excel(writer,sheet_name='不区分责任人刊登情况数据明细',index=False)


writer.save()
writer.close()

In [75]:
for i in set(path2['主体账号']):
    print(i)
    path2[path2['主体账号']==i].drop(labels=['国内仓sku'],axis=1).to_excel(r'E:\20240712-Temu\责任类目刊登情况统计2025-09-01\{0}-责任类目刊登情况统计{1}.xlsx'.format(i,datetime.date.today().strftime('%Y-%m-%d')),index=False)
    

nan
李金强
libby
杨静
刘子龙
